#ALGO TRADING WITH ANTIGRAVITY

##1.CONTEXT

This notebook represents a pivotal demonstration of the advanced capabilities offered by **ANTIGRAVITY**, our cutting-edge platform designed for sophisticated algorithmic trading strategy development and backtesting. In the dynamic and often unpredictable world of financial markets, the ability to rapidly prototype, test, and refine trading strategies is paramount. ANTIGRAVITY provides the robust infrastructure and intuitive tools necessary for quantitative analysts and traders to transform complex ideas into actionable, data-driven strategies.

At its core, this notebook leverages ANTIGRAVITY's powerful analytical engine to construct an interactive backtesting environment specifically tailored for the 30 components of the Dow Jones Industrial Average (DJIA). The DJIA, a venerable and widely followed market index, offers a compelling universe of blue-chip stocks for exploring various investment philosophies. This backtester is more than just a simulation; it's a testament to how ANTIGRAVITY empowers users to delve deep into market mechanics, evaluate hypothetical scenarios, and gain a profound understanding of strategy performance under historical market conditions.

We will explore a diverse array of algorithmic trading strategies, each meticulously implemented and rigorously tested within the ANTIGRAVITY framework. This includes the time-honored 'Dogs of the Dow' approach, which seeks to capitalize on dividend yield anomalies, as well as more complex methodologies like momentum (relative strength) and mean reversion. Furthermore, the notebook provides essential baselines such as equal-weighted and price-weighted portfolios, allowing for a comprehensive comparative analysis against traditional market representations.

One of the standout features showcased here is ANTIGRAVITY's seamless integration of both a pure Python backtesting engine—optimized for performance and precision—and an embedded, interactive web dashboard. This dual-pronged approach offers flexibility, enabling deep programmatic control for developers while simultaneously providing an engaging, user-friendly interface for visual analysis. The dashboard, running directly within the Colab environment, offers real-time playback of portfolio evolution, detailed KPI metrics, and dynamic visualizations of equity curves and drawdowns, bringing your backtesting results to life.

Through this demonstration, ANTIGRAVITY aims to illustrate its commitment to empowering financial professionals with the tools needed to navigate market complexities, mitigate risks, and uncover profitable opportunities. Whether you are a seasoned quant researcher, an aspiring algorithmic trader, or simply keen to understand the mechanics behind modern trading strategies, this notebook, powered by ANTIGRAVITY, offers an unparalleled educational and practical experience. Dive in, experiment with parameters, and witness firsthand the transformative potential of robust algorithmic design and backtesting.

## 2.HOW ANTIGRAVITY GENERATED THIS APPLICATION

This sophisticated algorithmic trading backtester, complete with its interactive web dashboard, was not built line-by-line through traditional manual coding but rather intelligently assembled and optimized by **ANTIGRAVITY**. Our platform leverages advanced AI and generative programming techniques to transform high-level specifications and user intents into functional, well-structured, and performant code.

At its core, ANTIGRAVITY operates on a principle of semantic understanding. When presented with a request such as 'build an interactive backtesting engine for DJIA components,' it first deconstructs the intent into core functionalities: data acquisition, strategy implementation, performance metrics calculation, and user interface generation. For data acquisition, ANTIGRAVITY intelligently identified the need for historical stock prices and dividends, interfacing with relevant financial data providers (like Yahoo Finance, as seen in the Python cells) and structuring the data for efficient processing.

The strategic components, such as 'Dogs of the Dow,' 'Momentum,' and 'Mean Reversion,' were interpreted as distinct algorithmic patterns. ANTIGRAVITY's generative engine then formulated the Python logic for each strategy, ensuring correct handling of rebalancing schedules, transaction costs, and portfolio allocation. It understood the intricacies of calculating dividend yields for 'Dogs' and momentum factors for 'Momentum,' translating these financial concepts into precise mathematical and programmatic steps.

Crucially, the interactive web dashboard was also a product of ANTIGRAVITY's generative capabilities. Recognizing the need for a dynamic and intuitive user experience, ANTIGRAVITY synthesized the HTML, CSS, and JavaScript required to create the responsive glassmorphic interface you see. This involved:

1.  **Component Generation:** Creating UI elements like dropdowns, sliders, and buttons, and linking them to underlying strategy parameters.
2.  **Visualization Integration:** Selecting appropriate charting libraries (Chart.js was chosen for its flexibility and performance within web environments) and generating the code to dynamically update equity curves, drawdowns, and portfolio allocations based on backtest results.
3.  **Real-time Playback Logic:** Implementing the JavaScript state machine to simulate the daily progression of the backtest, update KPIs, and render the portfolio composition timeline. This complex synchronization between Python backend results and JavaScript frontend animation was orchestrated by ANTIGRAVITY to provide a seamless interactive experience.
4.  **Colab Integration:** Understanding the execution environment (Google Colab) and generating the necessary Python code to serve the web application locally within an iframe, enabling direct interaction without external deployment.

In essence, ANTIGRAVITY acted as a highly intelligent, domain-aware co-pilot, translating conceptual requirements into tangible code artifacts across multiple languages and frameworks. This allowed for rapid prototyping and deployment of a sophisticated financial application, significantly reducing the development cycle and enabling a deeper focus on strategic insights rather than low-level implementation details.

In [8]:
# Install required dependencies
!pip install yfinance pandas numpy plotly portpicker

In [9]:
import yfinance as yf
import pandas as pd
import numpy as np
import json
import datetime

# Define the 30 active components of the Dow Jones Industrial Average
tickers = [
    'MMM', 'GOOGL', 'AMZN', 'AXP', 'AMGN', 'AAPL', 'BA', 'CAT', 'CVX', 'CSCO',
    'KO', 'DIS', 'SHW', 'GS', 'HD', 'HON', 'IBM', 'NVDA', 'JNJ', 'JPM',
    'MCD', 'MRK', 'MSFT', 'NKE', 'PG', 'CRM', 'TRV', 'UNH', 'V', 'WMT'
]

print("Downloading historical prices and dividends (from 2015 to current)...")
start_date = "2015-01-01"
end_date = datetime.datetime.now().strftime("%Y-%m-%d")

adj_close_df = pd.DataFrame()
close_df = pd.DataFrame()
dividends_dict = {}

for i, ticker in enumerate(tickers, 1):
    print(f"[{i}/{len(tickers)}] Downloading {ticker}...")
    try:
        t = yf.Ticker(ticker)
        hist = t.history(start=start_date, end=end_date, auto_adjust=False)
        if not hist.empty:
            if hist.index.tz is not None:
                hist.index = hist.index.tz_localize(None)

            adj_col = 'Adj Close' if 'Adj Close' in hist.columns else 'Close'
            adj_close_df[ticker] = hist[adj_col]
            close_df[ticker] = hist['Close']

            # Extract dividend distributions
            divs = hist[hist['Dividends'] > 0]
            dividends_dict[ticker] = [
                {"date": date.strftime('%Y-%m-%d'), "amount": float(row['Dividends'])}
                for date, row in divs.iterrows()
            ]
    except Exception as e:
        print(f"  Error downloading {ticker}: {e}")

# Forward fill and backward fill any missing values
adj_close_df = adj_close_df.ffill().bfill()
close_df = close_df.ffill().bfill()
dates_list = [date.strftime('%Y-%m-%d') for date in adj_close_df.index]

print(f"\nSUCCESS: Loaded {len(dates_list)} trading days for {len(adj_close_df.columns)} assets.")

[1/30] Downloading MMM...
[2/30] Downloading GOOGL...
[3/30] Downloading AMZN...
[4/30] Downloading AXP...
[5/30] Downloading AMGN...
[6/30] Downloading AAPL...
[7/30] Downloading BA...
[8/30] Downloading CAT...
[9/30] Downloading CVX...
[10/30] Downloading CSCO...
[11/30] Downloading KO...
[12/30] Downloading DIS...
[13/30] Downloading SHW...
[14/30] Downloading GS...
[15/30] Downloading HD...
[16/30] Downloading HON...
[17/30] Downloading IBM...
[18/30] Downloading NVDA...
[19/30] Downloading JNJ...
[20/30] Downloading JPM...
[21/30] Downloading MCD...
[22/30] Downloading MRK...
[23/30] Downloading MSFT...
[24/30] Downloading NKE...
[25/30] Downloading PG...
[26/30] Downloading CRM...
[27/30] Downloading TRV...
[28/30] Downloading UNH...
[29/30] Downloading V...
[30/30] Downloading WMT...

SUCCESS: Loaded 2896 trading days for 30 assets.


##3.PURE PYTHON BACKTESTER ENGINE

The 'Pure Python Backtesting Engine' outlined in this notebook provides a robust, transparent, and computationally efficient framework for simulating algorithmic trading strategies. Unlike the interactive web dashboard, this engine operates solely within the Python environment, offering fine-grained control and detailed output for analysis. Its design emphasizes modularity, allowing for easy integration of various strategies and performance metrics.

At its core, the `DJIABacktester` class is initialized with historical market data, including adjusted close prices, raw close prices, and dividend distributions for the selected DJIA components. This ensures that the simulation accurately reflects real-world market conditions, accounting for splits, dividends, and other corporate actions. The `run_strategy` method is the central orchestrator, accepting parameters such as the strategy type (`momentum`, `dogs`, `mean_reversion`, `equal_weight`, `price_weighted`), rebalancing frequency, number of stocks to hold, lookback periods, and crucial trading costs like initial capital and transaction fees. This parameterization allows for extensive experimentation and optimization of strategy logic.

Within the `run_strategy` method, the simulation proceeds day-by-day over the historical data. A key aspect is the `rebalance_indices` calculation, which meticulously identifies the exact trading days when the portfolio needs to be adjusted according to the chosen rebalancing frequency (monthly, quarterly, or annually). This ensures that strategies like 'Dogs of the Dow'—which are often rebalanced annually—are simulated correctly.

For each rebalancing event, the engine performs a three-phase process:

1.  **Selection Phase:** Based on the chosen strategy, assets are identified. For instance, the 'Dogs of the Dow' strategy dynamically calculates dividend yields based on historical distributions and selects the top N highest-yielding stocks. Momentum and mean-reversion strategies analyze past performance over a defined lookback period, optionally applying technical filters like a 200-day Simple Moving Average to confirm trends.
2.  **Weighting Phase:** Once assets are selected, their allocation within the portfolio is determined. Strategies like 'equal_weight' distribute capital uniformly, while 'price_weighted' mimics the DJIA's own mechanism, allocating more capital to higher-priced stocks.
3.  **Execution Phase:** This is where the 'rubber meets the road.' The engine calculates the target allocations versus current holdings, determines the necessary buy and sell orders, and critically, applies transaction costs. These costs, representing slippage and commissions, are deducted from the portfolio's cash, providing a more realistic assessment of strategy profitability. The `trade_log` captures every transaction, offering a forensic record of portfolio activity.

Throughout the simulation, the `equity_curve` is meticulously updated daily to reflect the portfolio's cumulative value. Furthermore, `daily_snapshots` record the exact holdings and their percentage allocations for each day, which is crucial for visualizing the portfolio's evolution. Finally, after the simulation completes, the `calculate_metrics` function processes the `equity_curve` to derive standard performance statistics such as Total Return, CAGR, Volatility, Sharpe Ratio, and Maximum Drawdown. This comprehensive output allows for a rigorous quantitative evaluation of the strategy's historical performance, providing the foundational data for the interactive visualizations.

In [15]:
class DJIABacktester:
    def __init__(self, dates, tickers, adj_close, close, dividends):
        self.dates = dates
        self.tickers = tickers
        self.adj_close = adj_close # DataFrame
        self.close = close # DataFrame
        self.dividends = dividends # dict

    def run_strategy(self, strategy='momentum', rebalance_freq='monthly',
                     stock_count=10, lookback_months=6, use_ma_filter=True,
                     initial_capital=100000.0, transaction_cost=0.001):

        dates = self.dates
        tickers = self.tickers
        prices = self.adj_close.values
        close_prices = self.close.values
        total_days = len(dates)
        ticker_to_idx = {t: idx for idx, t in enumerate(tickers)}

        # Precompute SMA 200
        sma_200 = {}
        if use_ma_filter:
            for t in tickers:
                series = self.adj_close[t].values
                sma = pd.Series(series).rolling(window=200, min_periods=1).mean().values
                sma_200[t] = sma

        # Resolve rebalancing indices
        rebalance_indices = [0]
        for d in range(1, total_days):
            prev_date = dates[d-1]
            curr_date = dates[d]

            prev_year = prev_date[0:4]
            curr_year = curr_date[0:4]
            prev_month = prev_date[5:7]
            curr_month = curr_date[5:7]

            should_rebal = False
            if rebalance_freq == 'monthly':
                should_rebal = (prev_month != curr_month)
            elif rebalance_freq == 'quarterly':
                prev_q = (int(prev_month) - 1) // 3
                curr_q = (int(curr_month) - 1) // 3
                should_rebal = (prev_q != curr_q)
            elif rebalance_freq == 'annually':
                should_rebal = (prev_year != curr_year)

            if should_rebal:
                rebalance_indices.append(d)

        # Portfolio simulation loop
        cash = initial_capital
        shares = np.zeros(len(tickers))
        equity_curve = []
        trade_log = []
        daily_snapshots = []

        rebalance_ptr = 0
        selected_assets = []

        for d in range(total_days):
            curr_date = dates[d]

            # Valuation
            stock_valuation = np.sum(shares * prices[d, :])
            total_value = cash + stock_valuation
            equity_curve.append(total_value)

            # Rebalance triggers
            if rebalance_ptr < len(rebalance_indices) and d == rebalance_indices[rebalance_ptr]:
                rebalance_ptr += 1

                # Strategy selections
                if strategy == 'equal_weight':
                    selected_assets = tickers.copy()
                elif strategy == 'price_weighted':
                    selected_assets = tickers.copy()
                elif strategy == 'dogs':
                    yields = []
                    rebal_dt = datetime.datetime.strptime(curr_date, '%Y-%m-%d')
                    one_year_ago = rebal_dt - datetime.timedelta(days=365)
                    for t in tickers:
                        divs = self.dividends.get(t, [])
                        div_sum = sum(e["amount"] for e in divs if one_year_ago <= datetime.datetime.strptime(e["date"], '%Y-%m-%d') <= rebal_dt)
                        current_close = close_prices[d, ticker_to_idx[t]]
                        yields.append((t, div_sum / current_close if current_close > 0 else 0))
                    yields.sort(key=lambda x: x[1], reverse=True)
                    selected_assets = [x[0] for x in yields[:stock_count]]
                elif strategy in ['momentum', 'mean_reversion']:
                    lookback_days = lookback_months * 21
                    lookback_idx = max(0, d - lookback_days)
                    performances = []
                    for t in tickers:
                        t_idx = ticker_to_idx[t]
                        start_p = prices[lookback_idx, t_idx]
                        end_p = prices[d, t_idx]
                        ret = (end_p - start_p) / start_p if start_p > 0 else 0
                        satisfies_ma = True
                        if use_ma_filter and d >= 200:
                            satisfies_ma = prices[d, t_idx] > sma_200[t][d]
                        performances.append((t, ret, satisfies_ma))

                    if strategy == 'momentum':
                        filtered = performances
                        if use_ma_filter:
                            filtered = [p for p in performances if p[2]]
                            if not filtered: filtered = performances
                        filtered.sort(key=lambda x: x[1], reverse=True)
                        selected_assets = [x[0] for x in filtered[:stock_count]]
                    else: # mean reversion
                        filtered = performances
                        filtered.sort(key=lambda x: x[1])
                        selected_assets = [x[0] for x in filtered[:stock_count]]

                # Weights calculations
                weights = np.zeros(len(tickers))
                if strategy == 'price_weighted':
                    price_sum = sum(close_prices[d, ticker_to_idx[t]] for t in selected_assets)
                    for t in selected_assets:
                        t_idx = ticker_to_idx[t]
                        weights[t_idx] = close_prices[d, t_idx] / price_sum if price_sum > 0 else 0
                else:
                    eq_w = 1.0 / len(selected_assets)
                    for t in selected_assets:
                        weights[ticker_to_idx[t]] = eq_w

                # Orders and costs
                target_alloc = weights * total_value
                curr_alloc = shares * prices[d, :]
                trade_diffs = target_alloc - curr_alloc

                trade_val = np.sum(np.abs(trade_diffs))
                total_tc = trade_val * transaction_cost
                investable = total_value - total_tc

                new_shares = (investable * weights) / prices[d, :]
                new_shares[np.isnan(new_shares) | np.isinf(new_shares)] = 0

                # Log trades
                if d > 0:
                    for idx, t in enumerate(tickers):
                        diff = trade_diffs[idx]
                        if abs(diff) > 1.0:
                            action = "BUY" if diff > 0 else "SELL"
                            trade_log.append({
                                "date": curr_date,
                                "ticker": t,
                                "action": action,
                                "price": prices[d, idx],
                                "shares": abs(diff) / prices[d, idx],
                                "value": abs(diff)
                            })

                shares = new_shares
                cash = total_value - np.sum(shares * prices[d, :]) - total_tc

            # Day Allocations snapshot
            day_holdings = []
            for idx, t in enumerate(tickers):
                val = shares[idx] * prices[d, idx]
                if val > 10.0:
                    day_holdings.append({"ticker": t, "percentage": (val / total_value) * 100})
            day_holdings.sort(key=lambda x: x["percentage"], reverse=True)
            daily_snapshots.append({"date": curr_date, "holdings": day_holdings, "assets": selected_assets.copy()})

        return {
            "dates": dates,
            "equity_curve": equity_curve,
            "trade_log": trade_log,
            "daily_snapshots": daily_snapshots,
            "rebalance_indices": rebalance_indices
        }

In [16]:
def calculate_metrics(dates, curve, rf=0.02):
    final_val = curve[-1]
    start_val = curve[0]
    total_return = (final_val - start_val) / start_val

    years = len(dates) / 252.0
    cagr = (final_val / start_val) ** (1.0 / years) - 1.0

    returns = pd.Series(curve).pct_change().dropna()
    vol = returns.std() * np.sqrt(252)

    sharpe = (cagr - rf) / vol if vol > 0 else 0

    # Drawdown
    peaks = pd.Series(curve).cummax()
    dds = (pd.Series(curve) - peaks) / peaks
    max_dd = dds.min()

    return {
        "total_return": total_return * 100,
        "cagr": cagr * 100,
        "volatility": vol * 100,
        "sharpe": sharpe,
        "max_dd": max_dd * 100
    }

In [17]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Initialize backtester
backtester = DJIABacktester(dates_list, tickers, adj_close_df, close_df, dividends_dict)

# 2. Run Momentum Strategy (10 stocks, monthly, 6-month lookback, 200-SMA check)
strat_results = backtester.run_strategy(strategy='momentum', rebalance_freq='monthly', stock_count=10, lookback_months=6, use_ma_filter=True)

# 3. Run Benchmark (Price Weighted DJIA proxy)
bench_results = backtester.run_strategy(strategy='price_weighted', rebalance_freq='annually', stock_count=30, use_ma_filter=False)

# 4. Calculate metrics
strat_metrics = calculate_metrics(dates_list, strat_results["equity_curve"])
bench_metrics = calculate_metrics(dates_list, bench_results["equity_curve"])

print("=== BACKTEST PERFORMANCE PERFORMANCE COMPARISON ===")
print(f"Strategy Return: {strat_metrics['total_return']:.2f}% | CAGR: {strat_metrics['cagr']:.2f}% | Sharpe: {strat_metrics['sharpe']:.2f} | Max DD: {strat_metrics['max_dd']:.2f}%")
print(f"Benchmark Return: {bench_metrics['total_return']:.2f}% | CAGR: {bench_metrics['cagr']:.2f}% | Sharpe: {bench_metrics['sharpe']:.2f} | Max DD: {bench_metrics['max_dd']:.2f}%")

# 5. Plot Performance Comparison
fig = go.Figure()
fig.add_trace(go.Scatter(x=dates_list, y=strat_results["equity_curve"], name="Momentum Strategy", line=dict(color='#10B981', width=2)))
fig.add_trace(go.Scatter(x=dates_list, y=bench_results["equity_curve"], name="DJIA Benchmark Index", line=dict(color='#3B82F6', width=1.5, dash='dash')))
fig.update_layout(
    title="Algorithmic Strategy vs DJIA Index (Cumulative Returns)",
    xaxis_title="Date",
    yaxis_title="Portfolio Value ($)",
    template="plotly_dark",
    hovermode="x unified"
)
fig.show()

=== BACKTEST PERFORMANCE PERFORMANCE COMPARISON ===
Strategy Return: 614.43% | CAGR: 18.66% | Sharpe: 0.88 | Max DD: -30.05%
Benchmark Return: 350.74% | CAGR: 14.00% | Sharpe: 0.69 | Max DD: -35.66%


In [18]:
# Calculate drawdown series
def get_dd_series(curve):
    peaks = pd.Series(curve).cummax()
    return ((pd.Series(curve) - peaks) / peaks * 100).values

strat_dd = get_dd_series(strat_results["equity_curve"])
bench_dd = get_dd_series(bench_results["equity_curve"])

fig_dd = go.Figure()
fig_dd.add_trace(go.Scatter(x=dates_list, y=strat_dd, fill='tozeroy', name="Strategy Drawdown", line=dict(color='#EF4444', width=1.5)))
fig_dd.add_trace(go.Scatter(x=dates_list, y=bench_dd, name="Benchmark Drawdown", line=dict(color='rgba(59, 130, 246, 0.4)', width=1, dash='dot')))
fig_dd.update_layout(
    title="Drawdown Underwater Risk Analysis (% Decline from Peak)",
    xaxis_title="Date",
    yaxis_title="Drawdown (%)",
    template="plotly_dark"
)
fig_dd.show()

## 4.THE DYNAMIC PORTFOLIO COMPOSITION LOG FEED


The 'Dynamic Portfolio Composition Log Feed' section is a crucial analytical tool for understanding the behavior of the algorithmic trading strategy over time. While the equity curve and drawdown charts provide a high-level view of performance, this log feed offers a granular, chronological breakdown of the portfolio's rebalancing activities.

Its primary purpose is to visualize the *entrants* and *exits* from the portfolio during each rebalancing event. As the backtesting engine progresses through time, based on the defined rebalancing frequency (e.g., monthly, quarterly, annually), the strategy selects new assets to hold and liquidates others. This log feed captures these changes:

*   **Date and Valuation:** Each entry is timestamped with the rebalancing date and displays the portfolio's total valuation at that specific point, providing context for the allocation decisions.
*   **'Joined' (Entrants):** This list shows the tickers of assets that were newly added to the portfolio during that rebalancing cycle. For strategies like 'Dogs of the Dow' or 'Momentum', these would be the top-ranking stocks according to the strategy's criteria at that time.
*   **'Left' (Exits):** Conversely, this list displays the tickers of assets that were removed or liquidated from the portfolio. These are typically stocks that no longer meet the strategy's selection criteria or have been replaced by higher-ranking assets.

This dynamic log feed is invaluable for several reasons:

1.  **Transparency:** It provides a clear, auditable trail of every portfolio adjustment, allowing users to understand *why* certain trades were made at specific times.
2.  **Strategy Validation:** By reviewing the entrants and exits, users can verify if the strategy is behaving as expected. For example, are 'Dogs of the Dow' truly selecting high-yielding stocks, or is the momentum strategy correctly identifying leading assets?
3.  **Risk Management Insight:** Frequent turnover or sudden shifts in portfolio composition can indicate certain risks or unintended biases in the strategy. This log helps in identifying such patterns.
4.  **Learning and Refinement:** Analyzing the historical rebalancing decisions can offer insights into market behavior and help in refining strategy parameters or even identifying new strategy ideas.

In the interactive web dashboard, this feed is animated, building dynamically as the backtest plays out, offering a powerful visual narrative of the strategy's execution and adaptation to changing market conditions.

In [19]:
rebal_indices = strat_results["rebalance_indices"]
snapshots = strat_results["daily_snapshots"]

print("=== HISTORICAL PORTFOLIO REBALANCING LOG STREAM ===\n")
for idx_idx, d_idx in enumerate(rebal_indices):
    date = snapshots[d_idx]["date"]
    curr_assets = snapshots[d_idx]["assets"]
    portfolio_val = strat_results["equity_curve"][d_idx]

    prev_assets = []
    if idx_idx > 0:
        prev_idx = rebal_indices[idx_idx - 1]
        prev_assets = snapshots[prev_idx]["assets"]

    entrants = [t for t in curr_assets if t not in prev_assets]
    exits = [t for t in prev_assets if t not in curr_assets]

    print(f"Date: {date} | Valuation: ${portfolio_val:,.2f}")
    print(f"  Joined: {entrants if entrants else 'None'}")
    print(f"  Left:   {exits if exits else 'None'}")
    print("-" * 80)

=== HISTORICAL PORTFOLIO REBALANCING LOG STREAM ===

Date: 2015-01-02 | Valuation: $100,000.00
  Joined: ['MMM', 'GOOGL', 'AMZN', 'AXP', 'AMGN', 'AAPL', 'BA', 'CAT', 'CVX', 'CSCO']
  Left:   None
--------------------------------------------------------------------------------
Date: 2015-02-02 | Valuation: $100,292.81
  Joined: ['UNH', 'MRK', 'SHW', 'HD', 'WMT']
  Left:   ['AXP', 'AMGN', 'CAT', 'CVX', 'CSCO']
--------------------------------------------------------------------------------
Date: 2015-03-02 | Valuation: $105,413.82
  Joined: ['CRM', 'DIS', 'NVDA', 'CSCO']
  Left:   ['MRK', 'GOOGL', 'MMM', 'WMT']
--------------------------------------------------------------------------------
Date: 2015-04-01 | Valuation: $101,970.63
  Joined: ['NKE']
  Left:   ['CSCO']
--------------------------------------------------------------------------------
Date: 2015-05-01 | Valuation: $105,273.54
  Joined: ['IBM']
  Left:   ['NKE']
----------------------------------------------------------------

##5.EMBEDDED INTERACTIVE DASHBOARD

Run this cell to start a local background server inside the Colab environment and display the full HTML/CSS/JS glassmorphic playback application in a Google Colab native port forwarding iframe. This delivers the complete interactive controls and animated curves directly within this notebook!

In [20]:
# 1. Save data structure as local JSON
json_data = {
    "dates": dates_list,
    "tickers": list(adj_close_df.columns),
    "prices": {t: [round(float(v), 4) for v in adj_close_df[t].values] for t in adj_close_df.columns},
    "close_prices": {t: [round(float(v), 4) for v in close_df[t].values] for t in close_df.columns},
    "dividends": dividends_dict
}
with open('djia_history.json', 'w') as f:
    json.dump(json_data, f)

# 2. Fetch HTML dashboard code from current environment assets (we will inject the raw HTML, CSS, JS text)
# Write index.html, style.css, and app.js strings into local environment:
import os
import portpicker
import threading
import socket
from google.colab import output
from IPython.display import HTML
import http.server
import socketserver

# Write Dashboard code to Colab directory (Injecting files directly)
html_code = """<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Dow Jones Industrial Average Algorithmic Strategy Backtester</title>
    <!-- Google Fonts: Outfit & Inter -->
    <link rel="preconnect" href="https://fonts.googleapis.com">
    <link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&family=Outfit:wght@400;500;600;700;800&display=swap" rel="stylesheet">
    <!-- FontAwesome for Icons -->
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css">
    <!-- Chart.js CDN -->
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <link rel="stylesheet" href="style.css">
</head>
<body>
    <div class="app-container">
        <!-- Header -->
        <header class="app-header">
            <div class="header-logo">
                <i class="fa-solid fa-chart-line logo-icon"></i>
                <div class="logo-text">
                    <h1>DOW JONES</h1>
                    <span>ALGORITHMIC STRATEGY BACKTESTER</span>
                </div>
            </div>
            <div class="header-status">
                <span class="status-badge" id="dataStatus">
                    <i class="fa-solid fa-circle-notch fa-spin"></i> Loading Stock Data...
                </span>
                <span class="date-badge">
                    <i class="fa-regular fa-calendar"></i> Data Period: 2015 – 2026
                </span>
            </div>
        </header>

        <!-- Main Workspace -->
        <main class="app-main">
            <!-- Sidebar / Controls -->
            <aside class="control-panel card">
                <div class="card-header">
                    <i class="fa-solid fa-sliders"></i>
                    <h2>Strategy Settings</h2>
                </div>

                <div class="control-group">
                    <label for="strategySelect"><i class="fa-solid fa-brain"></i> Strategy</label>
                    <select id="strategySelect">
                        <option value="dogs">Dogs of the Dow</option>
                        <option value="momentum" selected>Momentum (Relative Strength)</option>
                        <option value="mean_reversion">Mean Reversion (Worst Performers)</option>
                        <option value="equal_weight">Equal Weight Buy & Hold</option>
                        <option value="price_weighted">Price-Weighted Index (DJIA Proxy)</option>
                    </select>
                </div>

                <!-- Strategy parameters (Dynamic visibility) -->
                <div id="paramGroup" class="param-section">
                    <div class="control-group" id="stockCountGroup">
                        <label for="stockCount"><i class="fa-solid fa-list-ol"></i> Stocks to Hold (N)</label>
                        <div class="range-container">
                            <input type="range" id="stockCount" min="3" max="25" value="10">
                            <span class="range-val" id="stockCountVal">10</span>
                        </div>
                    </div>

                    <div class="control-group" id="lookbackGroup">
                        <label for="lookbackPeriod"><i class="fa-solid fa-clock-rotate-left"></i> Lookback Period (Months)</label>
                        <select id="lookbackPeriod">
                            <option value="1">1 Month</option>
                            <option value="3">3 Months</option>
                            <option value="6" selected>6 Months</option>
                            <option value="12">12 Months</option>
                        </select>
                    </div>

                    <div class="control-group" id="maFilterGroup">
                        <label class="checkbox-label" for="maFilter">
                            <input type="checkbox" id="maFilter" checked>
                            <span class="custom-checkbox"></span>
                            <span class="label-text">Require Price > 200-day SMA</span>
                        </label>
                    </div>
                </div>

                <div class="control-group">
                    <label for="rebalanceFreq"><i class="fa-solid fa-arrows-rotate"></i> Rebalancing Frequency</label>
                    <select id="rebalanceFreq">
                        <option value="monthly" selected>Monthly</option>
                        <option value="quarterly">Quarterly</option>
                        <option value="annually">Annually</option>
                    </select>
                </div>

                <div class="control-group">
                    <label for="initialCapital"><i class="fa-solid fa-wallet"></i> Initial Capital ($)</label>
                    <input type="number" id="initialCapital" value="100000" min="1000" step="1000">
                </div>

                <div class="control-group">
                    <label for="transactionCost"><i class="fa-solid fa-receipt"></i> Slippage & Commissions (%)</label>
                    <div class="range-container">
                        <input type="range" id="transactionCost" min="0" max="1" step="0.05" value="0.10">
                        <span class="range-val" id="transactionCostVal">0.10%</span>
                    </div>
                </div>

                <button id="runBtn" class="btn btn-primary">
                    <i class="fa-solid fa-play"></i> Run Backtest
                </button>
            </aside>

            <!-- Dashboard Content -->
            <section class="dashboard-content">
                <!-- KPI Grid -->
                <div class="kpi-grid">
                    <div class="kpi-card card">
                        <div class="kpi-icon positive"><i class="fa-solid fa-percent"></i></div>
                        <div class="kpi-info">
                            <span class="kpi-title">Total Return</span>
                            <h3 class="kpi-value positive" id="metricTotalReturn">0.00%</h3>
                            <span class="kpi-sub" id="benchmarkTotalReturn">Benchmark: 0.00%</span>
                        </div>
                    </div>

                    <div class="kpi-card card">
                        <div class="kpi-icon"><i class="fa-solid fa-chart-line"></i></div>
                        <div class="kpi-info">
                            <span class="kpi-title">Annual Return (CAGR)</span>
                            <h3 class="kpi-value" id="metricCAGR">0.00%</h3>
                            <span class="kpi-sub" id="benchmarkCAGR">Benchmark: 0.00%</span>
                        </div>
                    </div>

                    <div class="kpi-card card">
                        <div class="kpi-icon negative"><i class="fa-solid fa-arrow-trend-down"></i></div>
                        <div class="kpi-info">
                            <span class="kpi-title">Max Drawdown</span>
                            <h3 class="kpi-value negative" id="metricDrawdown">0.00%</h3>
                            <span class="kpi-sub" id="benchmarkDrawdown">Benchmark: 0.00%</span>
                        </div>
                    </div>

                    <div class="kpi-card card">
                        <div class="kpi-icon warning"><i class="fa-solid fa-scale-balanced"></i></div>
                        <div class="kpi-info">
                            <span class="kpi-title">Sharpe Ratio</span>
                            <h3 class="kpi-value" id="metricSharpe">0.00</h3>
                            <span class="kpi-sub" id="benchmarkSharpe">Benchmark: 0.00</span>
                        </div>
                    </div>
                </div>

                <!-- Playback Controls Card -->
                <div class="playback-panel card">
                    <div class="playback-controls">
                        <button id="playBtn" class="play-btn">
                            <i class="fa-solid fa-play"></i> <span>Play Backtest</span>
                        </button>
                        <button id="resetBtn" class="reset-btn">
                            <i class="fa-solid fa-rotate-left"></i> <span>Reset</span>
                        </button>
                        <div class="playback-speed">
                            <label for="speedRange"><i class="fa-solid fa-gauge-high"></i> Speed:</label>
                            <input type="range" id="speedRange" min="1" max="10" value="5">
                            <span id="speedVal">5x</span>
                        </div>
                        <div class="playback-progress-container">
                            <span id="currentPlaybackDate">2015-01-02</span>
                            <div class="progress-bar-bg">
                                <div id="playbackProgress" class="progress-bar-fill" style="width: 0%"></div>
                            </div>
                            <span id="endPlaybackDate">2026-07-13</span>
                        </div>
                    </div>
                </div>

                <!-- Charts Section -->
                <div class="chart-container card">
                    <div class="card-header chart-tabs">
                        <div class="header-title">
                            <i class="fa-solid fa-chart-area"></i>
                            <h2>Performance History</h2>
                        </div>
                        <div class="tab-buttons">
                            <button class="tab-btn active" data-chart="performance">Equity Curve</button>
                            <button class="tab-btn" data-chart="drawdown">Drawdown (%)</button>
                            <button class="tab-btn" data-chart="annual">Annual Returns</button>
                        </div>
                    </div>
                    <div class="chart-wrapper">
                        <canvas id="performanceChart"></canvas>
                    </div>
                </div>

                <!-- Bottom Grid: Allocation & Trade Log -->
                <div class="bottom-grid">
                    <!-- Current Portfolio Allocation -->
                    <div class="allocation-panel card">
                        <div class="card-header">
                            <i class="fa-solid fa-pie-chart"></i>
                            <h2>Final Portfolio Allocation</h2>
                        </div>
                        <div class="allocation-wrapper">
                            <div class="chart-mini-wrapper">
                                <canvas id="allocationChart"></canvas>
                            </div>
                            <div class="allocation-legend" id="allocationLegend">
                                <!-- Populated dynamically -->
                            </div>
                        </div>
                    </div>

                    <!-- Portfolio Rebalance Activity -->
                    <div class="composition-panel card">
                        <div class="card-header">
                            <i class="fa-solid fa-timeline"></i>
                            <h2>Rebalancing Timeline</h2>
                        </div>
                        <div class="timeline-feed-wrapper" id="timelineFeed">
                            <div class="timeline-placeholder">
                                <i class="fa-solid fa-chart-line placeholder-icon"></i>
                                <p>Rebalancing event timeline will build dynamically here during playback.</p>
                            </div>
                        </div>
                    </div>

                    <!-- Trade Log -->
                    <div class="tradelog-panel card">
                        <div class="card-header">
                            <i class="fa-solid fa-list-check"></i>
                            <h2>Recent Rebalancing Trades</h2>
                        </div>
                        <div class="table-wrapper">
                            <table id="tradeTable">
                                <thead>
                                    <tr>
                                        <th>Date</th>
                                        <th>Ticker</th>
                                        <th>Action</th>
                                        <th>Price</th>
                                        <th>Shares</th>
                                        <th>Value</th>
                                    </tr>
                                </thead>
                                <tbody id="tradeLogBody">
                                    <tr>
                                        <td colspan="6" class="empty-table">No trades recorded yet. Run backtest.</td>
                                    </tr>
                                </tbody>
                            </table>
                        </div>
                    </div>
                </div>
            </section>
        </main>

        <footer class="app-footer-bar">
            <p>&copy; 2026 Dow Jones Algorithmic Strategy Backtester. Powered by historical market data.</p>
        </footer>
    </div>

    <!-- Script references -->
    <script src="app.js"></script>
</body>
</html>
"""
css_code = """/* ==========================================================================
   DOW JONES STRATEGY BACKTESTER CSS DESIGN SYSTEM
   ========================================================================== */

/* --- CSS Variables & Color Tokens --- */
:root {
    --bg-main: #0B0F19;
    --bg-card: rgba(17, 24, 39, 0.7);
    --bg-card-hover: rgba(23, 33, 53, 0.85);
    --border-color: rgba(255, 255, 255, 0.08);
    --border-glow: rgba(16, 185, 129, 0.2);

    --text-primary: #F3F4F6;
    --text-secondary: #9CA3AF;
    --text-muted: #6B7280;

    --color-primary: #10B981; /* Emerald */
    --color-primary-hover: #059669;
    --color-primary-glow: rgba(16, 185, 129, 0.15);

    --color-secondary: #3B82F6; /* Royal Blue */
    --color-secondary-glow: rgba(59, 130, 246, 0.15);

    --color-danger: #EF4444; /* Coral/Red */
    --color-danger-glow: rgba(239, 68, 68, 0.15);

    --color-warning: #F59E0B; /* Amber/Yellow */

    --shadow-sm: 0 1px 2px 0 rgba(0, 0, 0, 0.05);
    --shadow-md: 0 4px 6px -1px rgba(0, 0, 0, 0.1), 0 2px 4px -1px rgba(0, 0, 0, 0.06);
    --shadow-lg: 0 10px 15px -3px rgba(0, 0, 0, 0.3), 0 4px 6px -2px rgba(0, 0, 0, 0.15);
    --shadow-glow: 0 0 20px 0 rgba(16, 185, 129, 0.1);

    --font-sans: 'Inter', sans-serif;
    --font-heading: 'Outfit', sans-serif;

    --border-radius-sm: 6px;
    --border-radius-md: 12px;
    --border-radius-lg: 16px;

    --transition-fast: 0.15s ease;
    --transition-normal: 0.25s cubic-bezier(0.4, 0, 0.2, 1);
}

/* --- Base Reset & Setup --- */
* {
    margin: 0;
    padding: 0;
    box-sizing: border-box;
}

body {
    background-color: var(--bg-main);
    color: var(--text-primary);
    font-family: var(--font-sans);
    line-height: 1.5;
    background-image:
        radial-gradient(at 0% 0%, rgba(59, 130, 246, 0.08) 0px, transparent 50%),
        radial-gradient(at 100% 0%, rgba(16, 185, 129, 0.06) 0px, transparent 50%),
        radial-gradient(at 50% 100%, rgba(17, 24, 39, 0.5) 0px, transparent 80%);
    background-attachment: fixed;
    min-height: 100vh;
    display: flex;
    flex-direction: column;
}

/* --- Layout Grid --- */
.app-container {
    display: flex;
    flex-direction: column;
    min-height: 100vh;
    padding: 1.5rem;
    max-width: 1600px;
    margin: 0 auto;
    width: 100%;
    gap: 1.5rem;
}

/* --- Header Styling --- */
.app-header {
    display: flex;
    justify-content: space-between;
    align-items: center;
    padding-bottom: 0.5rem;
    border-bottom: 1px solid var(--border-color);
}

.header-logo {
    display: flex;
    align-items: center;
    gap: 0.75rem;
}

.logo-icon {
    font-size: 2.25rem;
    color: var(--color-primary);
    text-shadow: 0 0 15px var(--color-primary-glow);
}

.logo-text h1 {
    font-family: var(--font-heading);
    font-size: 1.5rem;
    font-weight: 800;
    letter-spacing: -0.025em;
    background: linear-gradient(135deg, #FFF 30%, var(--text-secondary) 100%);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}

.logo-text span {
    font-size: 0.75rem;
    font-weight: 600;
    letter-spacing: 0.1em;
    color: var(--color-primary);
}

.header-status {
    display: flex;
    gap: 0.75rem;
}

.status-badge, .date-badge {
    background-color: var(--bg-card);
    border: 1px solid var(--border-color);
    padding: 0.5rem 0.85rem;
    border-radius: var(--border-radius-sm);
    font-size: 0.85rem;
    font-weight: 500;
    display: flex;
    align-items: center;
    gap: 0.5rem;
}

.status-badge {
    color: var(--color-warning);
    border-color: rgba(245, 158, 11, 0.2);
}

.status-badge.ready {
    color: var(--color-primary);
    border-color: rgba(16, 185, 129, 0.2);
}

.status-badge.error {
    color: var(--color-danger);
    border-color: rgba(239, 68, 68, 0.2);
}

.date-badge {
    color: var(--text-secondary);
}

/* --- Main Structure --- */
.app-main {
    display: grid;
    grid-template-columns: 320px 1fr;
    gap: 1.5rem;
    align-items: start;
    flex-grow: 1;
}

/* --- Premium Cards --- */
.card {
    background-color: var(--bg-card);
    backdrop-filter: blur(16px);
    -webkit-backdrop-filter: blur(16px);
    border: 1px solid var(--border-color);
    border-radius: var(--border-radius-lg);
    box-shadow: var(--shadow-lg);
    transition: transform var(--transition-normal), border-color var(--transition-normal), box-shadow var(--transition-normal);
    overflow: hidden;
}

.card:hover {
    border-color: rgba(255, 255, 255, 0.12);
}

.card-header {
    display: flex;
    align-items: center;
    gap: 0.75rem;
    padding: 1.25rem 1.5rem;
    border-bottom: 1px solid var(--border-color);
}

.card-header h2 {
    font-family: var(--font-heading);
    font-size: 1.15rem;
    font-weight: 600;
    color: var(--text-primary);
}

.card-header i {
    color: var(--text-secondary);
    font-size: 1.1rem;
}

/* --- Control Panel / Sidebar --- */
.control-panel {
    padding-bottom: 1.5rem;
}

.control-panel .card-header i {
    color: var(--color-primary);
}

.control-group {
    padding: 0.75rem 1.5rem;
    display: flex;
    flex-direction: column;
    gap: 0.5rem;
}

.control-group label {
    font-size: 0.85rem;
    font-weight: 500;
    color: var(--text-secondary);
    display: flex;
    align-items: center;
    gap: 0.5rem;
}

.control-group label i {
    font-size: 0.9rem;
    opacity: 0.7;
}

/* Input Fields & Selects */
select, input[type="number"] {
    background-color: rgba(0, 0, 0, 0.2);
    border: 1px solid var(--border-color);
    color: var(--text-primary);
    padding: 0.65rem 0.85rem;
    border-radius: var(--border-radius-sm);
    font-family: var(--font-sans);
    font-size: 0.9rem;
    width: 100%;
    outline: none;
    transition: border-color var(--transition-fast), box-shadow var(--transition-fast);
}

select:focus, input[type="number"]:focus {
    border-color: var(--color-primary);
    box-shadow: 0 0 0 3px var(--color-primary-glow);
}

select option {
    background-color: var(--bg-main);
    color: var(--text-primary);
}

/* Slider Controls */
.range-container {
    display: flex;
    align-items: center;
    gap: 0.75rem;
}

input[type="range"] {
    -webkit-appearance: none;
    appearance: none;
    width: 100%;
    height: 6px;
    border-radius: 3px;
    background: rgba(255, 255, 255, 0.1);
    outline: none;
}

input[type="range"]::-webkit-slider-thumb {
    -webkit-appearance: none;
    appearance: none;
    width: 16px;
    height: 16px;
    border-radius: 50%;
    background: var(--color-primary);
    cursor: pointer;
    box-shadow: 0 0 10px var(--color-primary-glow);
    transition: transform var(--transition-fast), background var(--transition-fast);
}

input[type="range"]::-webkit-slider-thumb:hover {
    transform: scale(1.2);
    background: var(--text-primary);
}

.range-val {
    font-size: 0.85rem;
    font-weight: 600;
    min-width: 45px;
    text-align: right;
    color: var(--color-primary);
}

/* Checkbox Style */
.checkbox-label {
    display: flex;
    flex-direction: row;
    align-items: center;
    position: relative;
    cursor: pointer;
    user-select: none;
    padding: 0.25rem 0;
}

.checkbox-label input {
    position: absolute;
    opacity: 0;
    cursor: pointer;
    height: 0;
    width: 0;
}

.custom-checkbox {
    height: 18px;
    width: 18px;
    background-color: rgba(0, 0, 0, 0.2);
    border: 1px solid var(--border-color);
    border-radius: 4px;
    margin-right: 10px;
    display: flex;
    justify-content: center;
    align-items: center;
    transition: background-color var(--transition-fast), border-color var(--transition-fast);
}

.checkbox-label:hover input ~ .custom-checkbox {
    border-color: var(--color-primary);
}

.checkbox-label input:checked ~ .custom-checkbox {
    background-color: var(--color-primary);
    border-color: var(--color-primary);
}

.custom-checkbox:after {
    content: "";
    display: none;
    width: 4px;
    height: 8px;
    border: solid white;
    border-width: 0 2px 2px 0;
    transform: rotate(45deg);
    margin-bottom: 2px;
}

.checkbox-label input:checked ~ .custom-checkbox:after {
    display: block;
}

.label-text {
    font-size: 0.85rem;
    color: var(--text-primary);
}

/* Dynamic Parameter Section */
.param-section {
    border-top: 1px solid var(--border-color);
    border-bottom: 1px solid var(--border-color);
    padding: 0.5rem 0;
    background-color: rgba(255, 255, 255, 0.01);
}

/* Run Button */
.btn {
    display: flex;
    justify-content: center;
    align-items: center;
    gap: 0.5rem;
    font-family: var(--font-heading);
    font-size: 0.95rem;
    font-weight: 600;
    padding: 0.85rem 1.5rem;
    border: none;
    border-radius: var(--border-radius-sm);
    cursor: pointer;
    transition: transform var(--transition-fast), background-color var(--transition-fast), box-shadow var(--transition-fast);
    width: calc(100% - 3rem);
    margin: 1.25rem 1.5rem 0.5rem 1.5rem;
}

.btn-primary {
    background-color: var(--color-primary);
    color: white;
    box-shadow: 0 4px 14px var(--color-primary-glow);
}

.btn-primary:hover {
    background-color: var(--color-primary-hover);
    transform: translateY(-2px);
    box-shadow: 0 6px 20px rgba(16, 185, 129, 0.3);
}

.btn-primary:active {
    transform: translateY(0);
}

/* --- Dashboard Main Content Area --- */
.dashboard-content {
    display: flex;
    flex-direction: column;
    gap: 1.5rem;
}

/* --- KPI Metric Cards --- */
.kpi-grid {
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 1rem;
}

.kpi-card {
    display: flex;
    align-items: center;
    padding: 1.25rem 1.5rem;
    gap: 1.25rem;
}

.kpi-icon {
    width: 48px;
    height: 48px;
    border-radius: var(--border-radius-md);
    background-color: var(--color-secondary-glow);
    color: var(--color-secondary);
    display: flex;
    justify-content: center;
    align-items: center;
    font-size: 1.25rem;
    flex-shrink: 0;
}

.kpi-icon.positive {
    background-color: var(--color-primary-glow);
    color: var(--color-primary);
}

.kpi-icon.negative {
    background-color: var(--color-danger-glow);
    color: var(--color-danger);
}

.kpi-icon.warning {
    background-color: rgba(245, 158, 11, 0.15);
    color: var(--color-warning);
}

.kpi-info {
    display: flex;
    flex-direction: column;
}

.kpi-title {
    font-size: 0.8rem;
    font-weight: 600;
    color: var(--text-muted);
    text-transform: uppercase;
    letter-spacing: 0.05em;
}

.kpi-value {
    font-family: var(--font-heading);
    font-size: 1.5rem;
    font-weight: 700;
    color: var(--text-primary);
    line-height: 1.2;
    margin: 0.15rem 0 0.25rem 0;
}

.kpi-value.positive {
    color: var(--color-primary);
}

.kpi-value.negative {
    color: var(--color-danger);
}

.kpi-sub {
    font-size: 0.75rem;
    color: var(--text-secondary);
    white-space: nowrap;
}

/* --- Large Chart Section --- */
.chart-tabs {
    justify-content: space-between;
    padding-top: 0.75rem;
    padding-bottom: 0.75rem;
}

.chart-tabs .header-title {
    display: flex;
    align-items: center;
    gap: 0.75rem;
}

.tab-buttons {
    display: flex;
    background-color: rgba(0, 0, 0, 0.3);
    border: 1px solid var(--border-color);
    padding: 0.2rem;
    border-radius: var(--border-radius-sm);
}

.tab-btn {
    background: transparent;
    border: none;
    color: var(--text-secondary);
    font-family: var(--font-sans);
    font-size: 0.8rem;
    font-weight: 600;
    padding: 0.4rem 1rem;
    border-radius: 4px;
    cursor: pointer;
    transition: background-color var(--transition-fast), color var(--transition-fast);
}

.tab-btn:hover {
    color: var(--text-primary);
}

.tab-btn.active {
    background-color: var(--bg-main);
    color: var(--color-primary);
    box-shadow: var(--shadow-sm);
}

.chart-wrapper {
    position: relative;
    width: 100%;
    height: 400px;
    padding: 1.5rem;
}

/* --- Bottom Section Grid --- */
.bottom-grid {
    display: grid;
    grid-template-columns: 320px 1fr 1.2fr;
    gap: 1.5rem;
}

/* --- Playback Panel Styling --- */
.playback-panel {
    padding: 0.85rem 1.5rem;
    margin-bottom: 0.5rem;
    background-color: var(--bg-card);
}

.playback-controls {
    display: flex;
    align-items: center;
    justify-content: space-between;
    flex-wrap: wrap;
    gap: 1rem;
    width: 100%;
}

.playback-controls button {
    display: flex;
    align-items: center;
    gap: 0.5rem;
    background-color: rgba(255, 255, 255, 0.05);
    border: 1px solid var(--border-color);
    color: var(--text-primary);
    padding: 0.5rem 1rem;
    border-radius: var(--border-radius-sm);
    cursor: pointer;
    font-weight: 600;
    font-size: 0.85rem;
    font-family: var(--font-heading);
    transition: transform var(--transition-fast), background-color var(--transition-fast), border-color var(--transition-fast);
}

.playback-controls button:hover {
    background-color: rgba(255, 255, 255, 0.1);
    border-color: rgba(255, 255, 255, 0.2);
    transform: translateY(-1px);
}

.playback-controls button:active {
    transform: translateY(0);
}

.playback-controls button.play-btn {
    background-color: var(--color-primary-glow);
    border-color: rgba(16, 185, 129, 0.3);
    color: var(--color-primary);
    min-width: 140px;
}

.playback-controls button.play-btn:hover {
    background-color: var(--color-primary);
    color: white;
}

.playback-controls button.play-btn.playing {
    background-color: var(--color-danger-glow);
    border-color: rgba(239, 68, 68, 0.3);
    color: var(--color-danger);
}

.playback-controls button.play-btn.playing:hover {
    background-color: var(--color-danger);
    color: white;
}

.playback-controls button:disabled {
    opacity: 0.4;
    cursor: not-allowed;
    transform: none !important;
    background-color: rgba(255, 255, 255, 0.02) !important;
    border-color: var(--border-color) !important;
    color: var(--text-muted) !important;
}

.playback-speed {
    display: flex;
    align-items: center;
    gap: 0.75rem;
    font-size: 0.85rem;
    color: var(--text-secondary);
}

.playback-speed input[type="range"] {
    width: 80px;
}

.playback-speed span {
    font-weight: 700;
    color: var(--color-primary);
    min-width: 30px;
}

.playback-progress-container {
    display: flex;
    align-items: center;
    gap: 1rem;
    flex-grow: 1;
    font-size: 0.8rem;
    color: var(--text-muted);
}

.playback-progress-container span {
    font-weight: 500;
    font-family: var(--font-sans);
}

#currentPlaybackDate {
    color: var(--text-primary);
    font-weight: 600;
}

.progress-bar-bg {
    flex-grow: 1;
    height: 6px;
    background-color: rgba(255, 255, 255, 0.05);
    border-radius: 3px;
    overflow: hidden;
    position: relative;
}

.progress-bar-fill {
    height: 100%;
    background-color: var(--color-primary);
    box-shadow: 0 0 8px var(--color-primary-glow);
    width: 0%;
    transition: width 0.1s linear;
}

/* --- Portfolio Rebalance Activity (Timeline Feed) --- */
.timeline-feed-wrapper {
    overflow-y: auto;
    height: 380px;
    padding: 1.25rem 1.5rem;
    position: relative;
    display: flex;
    flex-direction: column;
    gap: 1.5rem;
}

.timeline-feed-wrapper::before {
    content: "";
    position: absolute;
    left: 2.25rem;
    top: 0;
    bottom: 0;
    width: 2px;
    background-color: var(--border-color);
    z-index: 1;
}

.timeline-feed-wrapper::-webkit-scrollbar {
    width: 4px;
}

.timeline-feed-wrapper::-webkit-scrollbar-thumb {
    background-color: var(--border-color);
    border-radius: 2px;
}

.timeline-placeholder {
    display: flex;
    flex-direction: column;
    align-items: center;
    justify-content: center;
    height: 100%;
    color: var(--text-muted);
    text-align: center;
    padding: 2rem;
    gap: 1rem;
    z-index: 2;
}

.placeholder-icon {
    font-size: 2.5rem;
    opacity: 0.2;
}

.timeline-placeholder p {
    font-size: 0.85rem;
    line-height: 1.4;
    max-width: 220px;
}

.timeline-event-card {
    position: relative;
    padding-left: 2.5rem;
    display: flex;
    flex-direction: column;
    gap: 0.5rem;
    animation: slideInDown 0.35s cubic-bezier(0.16, 1, 0.3, 1) forwards;
    z-index: 2;
}

.timeline-node {
    position: absolute;
    left: 0.95rem;
    top: 0.25rem;
    width: 14px;
    height: 14px;
    border-radius: 50%;
    border: 3px solid var(--color-primary);
    background-color: var(--bg-main);
    box-shadow: 0 0 8px var(--color-primary-glow);
    z-index: 3;
    transition: transform 0.2s ease;
}

.timeline-event-card:hover .timeline-node {
    transform: scale(1.2);
}

.timeline-event-header {
    display: flex;
    justify-content: space-between;
    align-items: center;
}

.timeline-event-date {
    font-family: var(--font-heading);
    font-weight: 700;
    font-size: 0.9rem;
    color: var(--text-primary);
}

.timeline-event-value {
    font-size: 0.75rem;
    color: var(--text-secondary);
    background-color: rgba(255, 255, 255, 0.05);
    border: 1px solid var(--border-color);
    padding: 0.15rem 0.4rem;
    border-radius: 4px;
}

.timeline-event-body {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 1rem;
    background-color: rgba(255, 255, 255, 0.02);
    border: 1px solid var(--border-color);
    padding: 0.75rem;
    border-radius: var(--border-radius-md);
}

.timeline-column {
    display: flex;
    flex-direction: column;
    gap: 0.4rem;
}

.timeline-column h4 {
    font-size: 0.7rem;
    font-weight: 600;
    text-transform: uppercase;
    letter-spacing: 0.05em;
    color: var(--text-muted);
}

.timeline-column-list {
    display: flex;
    flex-wrap: wrap;
    gap: 0.3rem;
}

.empty-column-msg {
    font-size: 0.75rem;
    color: var(--text-muted);
    font-style: italic;
}

.ticker-badge {
    display: inline-flex;
    align-items: center;
    padding: 0.15rem 0.45rem;
    border-radius: 3px;
    font-family: var(--font-sans);
    font-size: 0.7rem;
    font-weight: 700;
    border: 1px solid var(--border-color);
}

.ticker-badge.entrant {
    color: var(--color-primary);
    border-color: rgba(16, 185, 129, 0.2);
    background-color: var(--color-primary-glow);
}

.ticker-badge.exit {
    color: var(--color-danger);
    border-color: rgba(239, 68, 68, 0.2);
    background-color: var(--color-danger-glow);
    text-decoration: line-through;
}

@keyframes slideInDown {
    from {
        opacity: 0;
        transform: translateY(-12px);
    }
    to {
        opacity: 1;
        transform: translateY(0);
    }
}


/* Portfolio Allocation Chart */
.allocation-wrapper {
    display: flex;
    flex-direction: column;
    align-items: center;
    padding: 1.5rem;
    height: 380px;
    justify-content: center;
}

.chart-mini-wrapper {
    position: relative;
    width: 180px;
    height: 180px;
    margin-bottom: 1.5rem;
}

.allocation-legend {
    width: 100%;
    display: grid;
    grid-template-columns: repeat(2, 1fr);
    gap: 0.5rem 1rem;
    max-height: 120px;
    overflow-y: auto;
    padding-right: 0.5rem;
}

.allocation-legend::-webkit-scrollbar {
    width: 4px;
}

.allocation-legend::-webkit-scrollbar-thumb {
    background-color: var(--border-color);
    border-radius: 2px;
}

.legend-item {
    display: flex;
    align-items: center;
    font-size: 0.75rem;
    gap: 0.5rem;
}

.legend-color {
    width: 10px;
    height: 10px;
    border-radius: 2px;
    flex-shrink: 0;
}

.legend-label {
    color: var(--text-secondary);
    white-space: nowrap;
    overflow: hidden;
    text-overflow: ellipsis;
}

.legend-val {
    font-weight: 700;
    color: var(--text-primary);
    margin-left: auto;
}

/* Trade Log Table */
.tradelog-panel {
    display: flex;
    flex-direction: column;
}

.table-wrapper {
    overflow-y: auto;
    height: 380px;
    border-radius: 0 0 var(--border-radius-lg) var(--border-radius-lg);
}

.table-wrapper::-webkit-scrollbar {
    width: 6px;
    height: 6px;
}

.table-wrapper::-webkit-scrollbar-track {
    background: transparent;
}

.table-wrapper::-webkit-scrollbar-thumb {
    background-color: var(--border-color);
    border-radius: 3px;
}

table {
    width: 100%;
    border-collapse: collapse;
    text-align: left;
    font-size: 0.85rem;
}

th {
    background-color: rgba(0, 0, 0, 0.2);
    position: sticky;
    top: 0;
    z-index: 10;
    color: var(--text-secondary);
    font-weight: 600;
    padding: 1rem 1.25rem;
    border-bottom: 1px solid var(--border-color);
    font-size: 0.8rem;
    text-transform: uppercase;
    letter-spacing: 0.05em;
}

td {
    padding: 0.85rem 1.25rem;
    border-bottom: 1px solid var(--border-color);
    color: var(--text-primary);
}

tr:hover td {
    background-color: rgba(255, 255, 255, 0.015);
}

.empty-table {
    text-align: center;
    color: var(--text-muted);
    padding: 4rem 0;
    font-style: italic;
}

/* Badge tags in table */
.badge {
    display: inline-block;
    padding: 0.2rem 0.5rem;
    border-radius: 4px;
    font-size: 0.75rem;
    font-weight: 700;
    text-transform: uppercase;
}

.badge-buy {
    background-color: var(--color-primary-glow);
    color: var(--color-primary);
    border: 1px solid rgba(16, 185, 129, 0.3);
}

.badge-sell {
    background-color: var(--color-danger-glow);
    color: var(--color-danger);
    border: 1px solid rgba(239, 68, 68, 0.3);
}

/* --- Footer Styling --- */
.app-footer-bar {
    text-align: center;
    padding: 1rem 0;
    border-top: 1px solid var(--border-color);
    font-size: 0.75rem;
    color: var(--text-muted);
}

/* --- Responsive Adjustments --- */
@media (max-width: 1200px) {
    .app-main {
        grid-template-columns: 1fr;
    }

    .control-panel {
        max-width: 100%;
    }

    .btn {
        width: calc(100% - 3rem);
    }
}

@media (max-width: 900px) {
    .kpi-grid {
        grid-template-columns: repeat(2, 1fr);
    }

    .bottom-grid {
        grid-template-columns: 1fr;
    }

    .allocation-wrapper {
        height: auto;
    }
}

@media (max-width: 600px) {
    .kpi-grid {
        grid-template-columns: 1fr;
    }
}
"""
js_code = """/* ==========================================================================
   DOW JONES STRATEGY BACKTESTER CORE LOGIC & ANIMATION ENGINE
   ========================================================================== */

// --- Global Application State ---
let stockData = null;
let performanceChart = null;
let allocationChart = null;
let currentChartType = 'performance'; // 'performance', 'drawdown', 'annual'
let backtestResults = null;
let benchmarkResults = null;

// --- Playback State Variables ---
let isPlaybackActive = false;
let playbackIndex = 0;
let playbackIntervalId = null;
let lastRebalanceIndexIndex = -1; // Tracks the index in rebalanceIndices array

// --- DOM Elements ---
const DOM = {
    strategySelect: document.getElementById('strategySelect'),
    rebalanceFreq: document.getElementById('rebalanceFreq'),
    stockCount: document.getElementById('stockCount'),
    stockCountVal: document.getElementById('stockCountVal'),
    lookbackPeriod: document.getElementById('lookbackPeriod'),
    maFilter: document.getElementById('maFilter'),
    initialCapital: document.getElementById('initialCapital'),
    transactionCost: document.getElementById('transactionCost'),
    transactionCostVal: document.getElementById('transactionCostVal'),
    runBtn: document.getElementById('runBtn'),
    dataStatus: document.getElementById('dataStatus'),

    // Parameters visibility container
    paramGroup: document.getElementById('paramGroup'),
    stockCountGroup: document.getElementById('stockCountGroup'),
    lookbackGroup: document.getElementById('lookbackGroup'),
    maFilterGroup: document.getElementById('maFilterGroup'),

    // Playback Controls
    playBtn: document.getElementById('playBtn'),
    resetBtn: document.getElementById('resetBtn'),
    speedRange: document.getElementById('speedRange'),
    speedVal: document.getElementById('speedVal'),
    currentPlaybackDate: document.getElementById('currentPlaybackDate'),
    endPlaybackDate: document.getElementById('endPlaybackDate'),
    playbackProgress: document.getElementById('playbackProgress'),

    // Portfolio Composition Panel
    timelineFeed: document.getElementById('timelineFeed'),

    // KPI elements
    metricTotalReturn: document.getElementById('metricTotalReturn'),
    metricCAGR: document.getElementById('metricCAGR'),
    metricDrawdown: document.getElementById('metricDrawdown'),
    metricSharpe: document.getElementById('metricSharpe'),

    benchmarkTotalReturn: document.getElementById('benchmarkTotalReturn'),
    benchmarkCAGR: document.getElementById('benchmarkCAGR'),
    benchmarkDrawdown: document.getElementById('benchmarkDrawdown'),
    benchmarkSharpe: document.getElementById('benchmarkSharpe'),

    // Tables & Logs
    tradeLogBody: document.getElementById('tradeLogBody'),
    allocationLegend: document.getElementById('allocationLegend'),

    // Chart canvas
    chartCanvas: document.getElementById('performanceChart'),
    allocationCanvas: document.getElementById('allocationChart'),

    // Tab buttons
    tabBtns: document.querySelectorAll('.tab-btn')
};

// --- Page Initialization ---
document.addEventListener('DOMContentLoaded', () => {
    setupEventListeners();
    loadStockData();
});

// --- Setup Event Listeners ---
function setupEventListeners() {
    // Sliders dynamic value display
    DOM.stockCount.addEventListener('input', (e) => {
        DOM.stockCountVal.textContent = e.target.value;
    });

    DOM.transactionCost.addEventListener('input', (e) => {
        DOM.transactionCostVal.textContent = parseFloat(e.target.value).toFixed(2) + '%';
    });

    // Strategy selection change -> toggle inputs
    DOM.strategySelect.addEventListener('change', updateParameterVisibility);

    // Run button
    DOM.runBtn.addEventListener('click', runAndRenderBacktests);

    // Playback controls
    DOM.playBtn.addEventListener('click', togglePlayback);
    DOM.resetBtn.addEventListener('click', resetPlayback);
    DOM.speedRange.addEventListener('input', (e) => {
        DOM.speedVal.textContent = e.target.value + 'x';
    });

    // Chart tabs
    DOM.tabBtns.forEach(btn => {
        btn.addEventListener('click', (e) => {
            DOM.tabBtns.forEach(b => b.classList.remove('active'));
            btn.classList.add('active');
            currentChartType = btn.getAttribute('data-chart');
            if (isPlaybackActive || playbackIndex > 0) {
                renderPlaybackCharts(playbackIndex);
            } else {
                renderMainChart();
            }
        });
    });
}

// --- Dynamic Parameter Controls ---
function updateParameterVisibility() {
    const strategy = DOM.strategySelect.value;

    if (strategy === 'dogs') {
        DOM.paramGroup.style.display = 'block';
        DOM.stockCountGroup.style.display = 'block';
        DOM.lookbackGroup.style.display = 'none';
        DOM.maFilterGroup.style.display = 'none';

        // Dogs of the Dow defaults to 10 stocks
        DOM.stockCount.value = 10;
        DOM.stockCountVal.textContent = '10';
        DOM.stockCount.max = 30;
    } else if (strategy === 'momentum' || strategy === 'mean_reversion') {
        DOM.paramGroup.style.display = 'block';
        DOM.stockCountGroup.style.display = 'block';
        DOM.lookbackGroup.style.display = 'block';
        DOM.maFilterGroup.style.display = 'block';
        DOM.stockCount.max = 25;
    } else {
        // Benchmarks (equal weight / price weighted) don't need parameters
        DOM.paramGroup.style.display = 'none';
    }
}

// --- Load JSON database ---
async function loadStockData() {
    try {
        const response = await fetch('djia_history.json');
        if (!response.ok) {
            throw new Error('Failed to load djia_history.json. Run download_data.py first.');
        }
        stockData = await response.json();

        // Update Status Badge
        DOM.dataStatus.className = 'status-badge ready';
        DOM.dataStatus.innerHTML = '<i class="fa-solid fa-circle-check"></i> System Ready';

        // Update End Date label in playback
        DOM.endPlaybackDate.textContent = stockData.dates[stockData.dates.length - 1];

        // Trigger Parameter setup
        updateParameterVisibility();

        // Run initial default backtest
        runAndRenderBacktests();
    } catch (error) {
        console.error('Data loading error:', error);
        DOM.dataStatus.className = 'status-badge error';
        DOM.dataStatus.innerHTML = '<i class="fa-solid fa-circle-exclamation"></i> Data File Missing';

        DOM.tradeLogBody.innerHTML = `
            <tr>
                <td colspan="6" class="empty-table" style="color: var(--color-danger)">
                    <i class="fa-solid fa-triangle-exclamation"></i> Data database not found.<br>
                    Please execute the <code>download_data.py</code> script to fetch Yahoo Finance data.
                </td>
            </tr>
        `;
    }
}

// --- Backtest Simulation Engine ---
function runAndRenderBacktests() {
    if (!stockData) return;

    // Stop any active animation
    stopPlayback();
    playbackIndex = 0;
    lastRebalanceIndexIndex = -1;
    updatePlaybackUIReset();

    // 1. Gather UI parameters
    const strategy = DOM.strategySelect.value;
    const rebalanceFreq = DOM.rebalanceFreq.value;
    const stockCount = parseInt(DOM.stockCount.value);
    const lookbackMonths = parseInt(DOM.lookbackPeriod.value);
    const useMaFilter = DOM.maFilter.checked && (strategy === 'momentum' || strategy === 'mean_reversion');
    const initialCapital = parseFloat(DOM.initialCapital.value) || 100000;
    const transactionCost = (parseFloat(DOM.transactionCost.value) || 0) / 100.0;

    // 2. Run Strategy Backtest
    backtestResults = runStrategy({
        strategy,
        rebalanceFreq,
        stockCount,
        lookbackMonths,
        useMaFilter,
        initialCapital,
        transactionCost
    });

    // 3. Run Benchmark (Price-Weighted Index)
    benchmarkResults = runStrategy({
        strategy: 'price_weighted',
        rebalanceFreq: 'annually', // minimal rebalancing for benchmark
        stockCount: 30,
        lookbackMonths: 6,
        useMaFilter: false,
        initialCapital,
        transactionCost: 0 // no costs for theoretical index
    });

    // 4. Update UI KPIs
    updateKPIs(backtestResults, benchmarkResults);

    // 5. Render Main Chart (Equity Curve / Drawdown / Annual)
    renderMainChart();

    // 6. Render Allocation Pie Chart
    renderAllocationChart(backtestResults);

    // 7. Update Trade Table
    renderTradeLog(backtestResults);

    // 8. Render Full Rebalancing Timeline
    renderFullTimeline();
}

// --- Backtesting Solver Function ---
function runStrategy(params) {
    const dates = stockData.dates;
    const tickers = stockData.tickers;
    const prices = stockData.prices; // Adjusted close prices (total returns)
    const closePrices = stockData.close_prices; // Raw close prices (dividend/split unadjusted)
    const dividends = stockData.dividends;

    const totalDays = dates.length;

    // Setup Portfolio State
    let cash = params.initialCapital;
    let shares = {}; // ticker -> number of shares
    tickers.forEach(t => shares[t] = 0);

    let equityCurve = [];
    let tradeLog = [];
    let dailySnapshots = []; // New timeline tracker

    // Pre-calculate moving averages if needed
    let sma200 = {};
    if (params.useMaFilter) {
        tickers.forEach(ticker => {
            sma200[ticker] = new Array(totalDays).fill(0);
            let sum = 0;
            const p = prices[ticker];
            for (let d = 0; d < totalDays; d++) {
                sum += p[d];
                if (d >= 200) {
                    sum -= p[d - 200];
                    sma200[ticker][d] = sum / 200;
                } else {
                    sma200[ticker][d] = sum / (d + 1);
                }
            }
        });
    }

    // Identify Rebalancing Dates
    // We rebalance on the first day, and then when the period (month/quarter/year) changes.
    let rebalanceIndices = [0];
    for (let d = 1; d < totalDays; d++) {
        const prevDate = dates[d - 1];
        const currDate = dates[d];

        const prevYear = prevDate.substring(0, 4);
        const currYear = currDate.substring(0, 4);
        const prevMonth = prevDate.substring(5, 7);
        const currMonth = currDate.substring(5, 7);

        let shouldRebalance = false;

        if (params.rebalanceFreq === 'monthly') {
            shouldRebalance = (prevMonth !== currMonth);
        } else if (params.rebalanceFreq === 'quarterly') {
            const prevQ = Math.floor((parseInt(prevMonth) - 1) / 3);
            const currQ = Math.floor((parseInt(currMonth) - 1) / 3);
            shouldRebalance = (prevQ !== currQ);
        } else if (params.rebalanceFreq === 'annually') {
            shouldRebalance = (prevYear !== currYear);
        }

        if (shouldRebalance) {
            rebalanceIndices.push(d);
        }
    }

    // Run Daily Simulation
    let rebalancePointer = 0;
    let selectedAssets = []; // Hold selected tickers at any day

    for (let d = 0; d < totalDays; d++) {
        const currentDateStr = dates[d];

        // 1. Calculate Portfolio Value at current day's prices
        let stockValuation = 0;
        tickers.forEach(ticker => {
            stockValuation += shares[ticker] * prices[ticker][d];
        });
        let totalValue = cash + stockValuation;
        equityCurve.push({ date: currentDateStr, value: totalValue });

        // 2. Perform Rebalancing if current index is a rebalancing date
        if (d === rebalanceIndices[rebalancePointer]) {
            rebalancePointer++;

            // Selection phase: Select assets based on Strategy
            if (params.strategy === 'equal_weight') {
                selectedAssets = [...tickers];
            } else if (params.strategy === 'price_weighted') {
                selectedAssets = [...tickers];
            } else if (params.strategy === 'dogs') {
                // Dogs of the Dow selection logic
                let yields = [];
                tickers.forEach(ticker => {
                    const divEvents = dividends[ticker] || [];
                    const currentClose = closePrices[ticker][d];

                    // Sum dividends in last 365 days
                    const rebalanceDate = new Date(currentDateStr);
                    const oneYearAgo = new Date(rebalanceDate.getTime() - (365 * 24 * 60 * 60 * 1000));

                    let divSum = 0;
                    divEvents.forEach(e => {
                        const ed = new Date(e.date);
                        if (ed >= oneYearAgo && ed <= rebalanceDate) {
                            divSum += e.amount;
                        }
                    });

                    const divYield = currentClose > 0 ? divSum / currentClose : 0;
                    yields.push({ ticker, yield: divYield });
                });

                yields.sort((a, b) => b.yield - a.yield);
                selectedAssets = yields.slice(0, params.stockCount).map(y => y.ticker);

            } else if (params.strategy === 'momentum' || params.strategy === 'mean_reversion') {
                const lookbackDays = params.lookbackMonths * 21;
                const lookbackIdx = Math.max(0, d - lookbackDays);

                let performances = [];

                tickers.forEach(ticker => {
                    let satisfiesMA = true;
                    if (params.useMaFilter && d >= 200) {
                        satisfiesMA = prices[ticker][d] > sma200[ticker][d];
                    }

                    const startPrice = prices[ticker][lookbackIdx];
                    const endPrice = prices[ticker][d];
                    const returnRate = (endPrice - startPrice) / startPrice;

                    performances.push({ ticker, returnRate, satisfiesMA });
                });

                if (params.strategy === 'momentum') {
                    let filtered = performances;
                    if (params.useMaFilter) {
                        filtered = performances.filter(p => p.satisfiesMA);
                        if (filtered.length === 0) filtered = performances;
                    }

                    filtered.sort((a, b) => b.returnRate - a.returnRate);
                    selectedAssets = filtered.slice(0, params.stockCount).map(p => p.ticker);
                } else {
                    let filtered = performances;
                    if (params.useMaFilter) {
                        filtered = performances.filter(p => p.satisfiesMA);
                        if (filtered.length === 0) filtered = performances;
                    }

                    filtered.sort((a, b) => a.returnRate - b.returnRate);
                    selectedAssets = filtered.slice(0, params.stockCount).map(p => p.ticker);
                }
            }

            // Weighting phase
            let weights = {};
            if (params.strategy === 'price_weighted') {
                let priceSum = 0;
                selectedAssets.forEach(ticker => {
                    priceSum += closePrices[ticker][d];
                });

                selectedAssets.forEach(ticker => {
                    weights[ticker] = priceSum > 0 ? closePrices[ticker][d] / priceSum : 0;
                });
            } else {
                const equalWeight = 1.0 / selectedAssets.length;
                selectedAssets.forEach(ticker => {
                    weights[ticker] = equalWeight;
                });
            }

            // Trading Execution Phase
            let targetAllocations = {};
            tickers.forEach(ticker => {
                targetAllocations[ticker] = weights[ticker] || 0;
            });

            let currentAllocations = {};
            tickers.forEach(ticker => {
                const stockVal = shares[ticker] * prices[ticker][d];
                currentAllocations[ticker] = totalValue > 0 ? stockVal / totalValue : 0;
            });

            let tradesToExecute = [];
            let totalTradeValue = 0;

            tickers.forEach(ticker => {
                const targetValue = totalValue * targetAllocations[ticker];
                const currentValue = shares[ticker] * prices[ticker][d];
                const tradeDiff = targetValue - currentValue;

                if (Math.abs(tradeDiff) > 1.0) {
                    tradesToExecute.push({
                        ticker,
                        currentValue,
                        targetValue,
                        diff: tradeDiff,
                        price: prices[ticker][d]
                    });
                    totalTradeValue += Math.abs(tradeDiff);
                }
            });

            const totalTransactionCosts = totalTradeValue * params.transactionCost;
            const investableCapital = totalValue - totalTransactionCosts;

            let newShares = {};
            tickers.forEach(ticker => {
                const tickerTargetVal = investableCapital * targetAllocations[ticker];
                newShares[ticker] = prices[ticker][d] > 0 ? tickerTargetVal / prices[ticker][d] : 0;
            });

            if (d > 0) {
                tradesToExecute.forEach(t => {
                    const action = t.diff > 0 ? 'BUY' : 'SELL';
                    const diffShares = Math.abs(t.diff) / t.price;

                    tradeLog.push({
                        date: currentDateStr,
                        ticker: t.ticker,
                        action,
                        price: t.price,
                        shares: diffShares,
                        value: Math.abs(t.diff)
                    });
                });
            }

            shares = newShares;
            let finalStockValue = 0;
            tickers.forEach(ticker => {
                finalStockValue += shares[ticker] * prices[ticker][d];
            });
            cash = totalValue - finalStockValue - totalTransactionCosts;
        }

        // Track current day holdings allocations (for animation playback)
        let dayHoldings = [];
        tickers.forEach(ticker => {
            const stockVal = shares[ticker] * prices[ticker][d];
            if (stockVal > 10.0) { // Positive allocation filter
                dayHoldings.push({
                    ticker,
                    value: stockVal,
                    percentage: totalValue > 0 ? (stockVal / totalValue) * 100 : 0
                });
            }
        });
        dayHoldings.sort((a, b) => b.value - a.value);

        dailySnapshots.push({
            date: currentDateStr,
            value: totalValue,
            assets: [...selectedAssets], // holds list of currently selected tickers
            holdings: dayHoldings
        });
    }

    // Performance Statistics Solver
    const finalValue = equityCurve[equityCurve.length - 1].value;
    const totalReturn = (finalValue - params.initialCapital) / params.initialCapital;

    const years = totalDays / 252.0;
    const cagr = Math.pow(finalValue / params.initialCapital, 1.0 / years) - 1.0;

    let dailyReturns = [];
    for (let i = 1; i < equityCurve.length; i++) {
        dailyReturns.push((equityCurve[i].value - equityCurve[i - 1].value) / equityCurve[i - 1].value);
    }

    let meanReturn = dailyReturns.reduce((sum, val) => sum + val, 0) / dailyReturns.length;
    let variance = dailyReturns.reduce((sum, val) => sum + Math.pow(val - meanReturn, 2), 0) / (dailyReturns.length - 1);
    let dailyVol = Math.sqrt(variance);
    let annualVol = dailyVol * Math.sqrt(252);

    const rf = 0.02;
    const sharpe = annualVol > 0 ? (cagr - rf) / annualVol : 0;

    let maxDrawdown = 0;
    let peak = -Infinity;
    let drawdowns = [];

    equityCurve.forEach(p => {
        if (p.value > peak) peak = p.value;
        const dd = peak > 0 ? (p.value - peak) / peak : 0;
        drawdowns.push({ date: p.date, drawdown: dd * 100 });
        if (dd < maxDrawdown) maxDrawdown = dd;
    });

    let annualReturns = {};
    equityCurve.forEach(p => {
        const year = p.date.substring(0, 4);
        if (!annualReturns[year]) {
            annualReturns[year] = { start: p.value, end: p.value };
        }
        annualReturns[year].end = p.value;
    });

    let yearsList = Object.keys(annualReturns).sort();
    yearsList.forEach((yr, idx) => {
        if (idx === 0) {
            annualReturns[yr].start = params.initialCapital;
        } else {
            const prevYr = yearsList[idx - 1];
            annualReturns[yr].start = annualReturns[prevYr].end;
        }
    });

    let annualReturnsList = yearsList.map(yr => {
        const start = annualReturns[yr].start;
        const end = annualReturns[yr].end;
        return {
            year: yr,
            returnRate: ((end - start) / start) * 100
        };
    });

    const positiveYears = annualReturnsList.filter(y => y.returnRate > 0).length;
    const winRate = annualReturnsList.length > 0 ? positiveYears / annualReturnsList.length : 0;

    let finalAllocations = [];
    tickers.forEach(ticker => {
        const finalVal = shares[ticker] * prices[ticker][totalDays - 1];
        if (finalVal > 10.0) {
            finalAllocations.push({
                ticker,
                value: finalVal,
                percentage: (finalVal / finalValue) * 100
            });
        }
    });
    finalAllocations.sort((a, b) => b.value - a.value);

    return {
        initialCapital: params.initialCapital,
        finalValue,
        totalReturn: totalReturn * 100,
        cagr: cagr * 100,
        annualVol: annualVol * 100,
        sharpe,
        maxDrawdown: maxDrawdown * 100,
        winRate: winRate * 100,
        equityCurve,
        drawdowns,
        annualReturns: annualReturnsList,
        tradeLog,
        finalAllocations,
        dailySnapshots,
        rebalanceIndices
    };
}

// --- Playback State Machine Engine ---
function togglePlayback() {
    if (!backtestResults) return;

    if (isPlaybackActive) {
        // Pause playback
        pausePlayback();
    } else {
        // Start/Resume playback
        startPlayback();
    }
}

function startPlayback() {
    isPlaybackActive = true;
    DOM.playBtn.innerHTML = '<i class="fa-solid fa-pause"></i> <span>Pause Backtest</span>';
    DOM.playBtn.className = 'play-btn playing';
    DOM.resetBtn.disabled = false;

    // Disable inputs while running to preserve integrity
    toggleInputControls(true);

    // Reset indices if we are at the end
    if (playbackIndex >= stockData.dates.length - 1) {
        playbackIndex = 0;
        lastRebalanceIndexIndex = -1;
        resetCompositionPanel();
    }

    // Loop step
    playbackIntervalId = setInterval(stepPlayback, 40);
}

function pausePlayback() {
    isPlaybackActive = false;
    DOM.playBtn.innerHTML = '<i class="fa-solid fa-play"></i> <span>Play Backtest</span>';
    DOM.playBtn.className = 'play-btn';
    clearInterval(playbackIntervalId);
}

function stopPlayback() {
    pausePlayback();
    toggleInputControls(false);
}

function resetPlayback() {
    stopPlayback();
    playbackIndex = 0;
    lastRebalanceIndexIndex = -1;
    updatePlaybackUIReset();

    // Draw full static curves
    renderMainChart();
    renderAllocationChart(backtestResults);
    renderTradeLog(backtestResults);
    resetCompositionPanel();
}

function stepPlayback() {
    const dates = stockData.dates;
    const total = dates.length;

    // Custom Speed Multipliers: Speed scale 1 to 10
    const speed = parseInt(DOM.speedRange.value);
    let step = 2; // base step size
    if (speed > 1) step = speed * 3; // speed 5 => step 15, speed 10 => step 30

    playbackIndex = Math.min(total - 1, playbackIndex + step);

    // Update Slider Progress elements
    const progress = (playbackIndex / (total - 1)) * 100;
    DOM.playbackProgress.style.width = progress + '%';
    DOM.currentPlaybackDate.textContent = dates[playbackIndex];

    // Render dynamic frames
    renderPlaybackCharts(playbackIndex);

    // Detect crossed rebalances to update entrants and exits
    const rebalanceIndices = backtestResults.rebalanceIndices;

    let activeRebalancePos = -1;
    for (let i = 0; i < rebalanceIndices.length; i++) {
        if (rebalanceIndices[i] <= playbackIndex) {
            activeRebalancePos = i;
        } else {
            break;
        }
    }

    if (activeRebalancePos > lastRebalanceIndexIndex) {
        // Remove placeholder if it exists on first card insert
        const placeholder = DOM.timelineFeed.querySelector('.timeline-placeholder');
        if (placeholder) placeholder.remove();

        // Render all newly crossed rebalance cards
        for (let r = lastRebalanceIndexIndex + 1; r <= activeRebalancePos; r++) {
            renderDynamicCompositionCard(r);
        }
        lastRebalanceIndexIndex = activeRebalancePos;
    }

    // End of playback check
    if (playbackIndex >= total - 1) {
        stopPlayback();
        DOM.playBtn.innerHTML = '<i class="fa-solid fa-play"></i> <span>Play Backtest</span>';
        DOM.playBtn.className = 'play-btn';
    }
}

// Disable/enable parameters during playback
function toggleInputControls(disable) {
    DOM.strategySelect.disabled = disable;
    DOM.rebalanceFreq.disabled = disable;
    DOM.stockCount.disabled = disable;
    DOM.lookbackPeriod.disabled = disable;
    DOM.maFilter.disabled = disable;
    DOM.initialCapital.disabled = disable;
    DOM.transactionCost.disabled = disable;
    DOM.runBtn.disabled = disable;
}

function updatePlaybackUIReset() {
    DOM.playbackProgress.style.width = '0%';
    DOM.currentPlaybackDate.textContent = stockData.dates[0];
    DOM.playBtn.innerHTML = '<i class="fa-solid fa-play"></i> <span>Play Backtest</span>';
    DOM.playBtn.className = 'play-btn';
    DOM.resetBtn.disabled = true;
    toggleInputControls(false);
}

// Reset Portfolio composition card to blank state
function resetCompositionPanel() {
    DOM.timelineFeed.innerHTML = `
        <div class="timeline-placeholder">
            <i class="fa-solid fa-chart-line placeholder-icon"></i>
            <p>Rebalancing event timeline will build dynamically here during playback.</p>
        </div>
    `;
}

// Render a single rebalance event card and prepend it to the feed
function renderDynamicCompositionCard(rebalanceIdxIdx) {
    const rebalanceIndices = backtestResults.rebalanceIndices;
    const dailySnapshots = backtestResults.dailySnapshots;

    const currIdx = rebalanceIndices[rebalanceIdxIdx];
    const currSnapshot = dailySnapshots[currIdx];
    const currAssets = currSnapshot.assets;
    const date = currSnapshot.date;
    const portfolioVal = currSnapshot.value;

    let prevAssets = [];
    if (rebalanceIdxIdx > 0) {
        const prevIdx = rebalanceIndices[rebalanceIdxIdx - 1];
        prevAssets = dailySnapshots[prevIdx].assets;
    }

    // Entrants: in current but not in previous
    const entrants = currAssets.filter(t => !prevAssets.includes(t));

    // Exits: in previous but not in current
    const exits = prevAssets.filter(t => !currAssets.includes(t));

    const card = document.createElement('div');
    card.className = 'timeline-event-card';

    // Circle Node
    const node = document.createElement('div');
    node.className = 'timeline-node';
    card.appendChild(node);

    // Header (Date & Portfolio Value)
    const header = document.createElement('div');
    header.className = 'timeline-event-header';
    header.innerHTML = `
        <span class="timeline-event-date">${date}</span>
        <span class="timeline-event-value">$${Math.round(portfolioVal).toLocaleString()}</span>
    `;
    card.appendChild(header);

    // Body (Entrants vs Exits columns)
    const body = document.createElement('div');
    body.className = 'timeline-event-body';

    // Entrants Column
    const entrantsCol = document.createElement('div');
    entrantsCol.className = 'timeline-column';
    const entrantsListHTML = entrants.length === 0
        ? '<span class="empty-column-msg">None</span>'
        : entrants.map(t => `<span class="ticker-badge entrant">${t}</span>`).join('');
    entrantsCol.innerHTML = `
        <h4>Joined</h4>
        <div class="timeline-column-list">${entrantsListHTML}</div>
    `;
    body.appendChild(entrantsCol);

    // Exits Column
    const exitsCol = document.createElement('div');
    exitsCol.className = 'timeline-column';
    const exitsListHTML = exits.length === 0
        ? '<span class="empty-column-msg">None</span>'
        : exits.map(t => `<span class="ticker-badge exit">${t}</span>`).join('');
    exitsCol.innerHTML = `
        <h4>Left</h4>
        <div class="timeline-column-list">${exitsListHTML}</div>
    `;
    body.appendChild(exitsCol);

    card.appendChild(body);

    // Prepend to feed
    DOM.timelineFeed.insertBefore(card, DOM.timelineFeed.firstChild);
}

// Render the entire rebalancing history feed in one go (for static overview)
function renderFullTimeline() {
    DOM.timelineFeed.innerHTML = "";
    const rebalanceIndices = backtestResults.rebalanceIndices;

    if (rebalanceIndices.length === 0) {
        DOM.timelineFeed.innerHTML = `
            <div class="timeline-placeholder">
                <i class="fa-solid fa-chart-line placeholder-icon"></i>
                <p>No rebalancing events recorded for this strategy.</p>
            </div>
        `;
        return;
    }

    // Loop chronologically. Prepending results in the newest rebalance card at the top.
    for (let r = 0; r < rebalanceIndices.length; r++) {
        renderDynamicCompositionCard(r);
    }
}

// Render dynamic daily slices on the charts
function renderPlaybackCharts(index) {
    const datesSlice = stockData.dates.slice(0, index + 1);

    // 1. Update line chart
    if (currentChartType === 'performance') {
        const strategyCurveSlice = backtestResults.equityCurve.slice(0, index + 1).map(p => p.value);
        const benchmarkCurveSlice = benchmarkResults.equityCurve.slice(0, index + 1).map(p => p.value);

        performanceChart.data.labels = datesSlice;
        performanceChart.data.datasets[0].data = strategyCurveSlice;
        performanceChart.data.datasets[1].data = benchmarkCurveSlice;
        performanceChart.update('none');

    } else if (currentChartType === 'drawdown') {
        const strategyDDSlice = backtestResults.drawdowns.slice(0, index + 1).map(d => d.drawdown);
        const benchmarkDDSlice = benchmarkResults.drawdowns.slice(0, index + 1).map(d => d.drawdown);

        performanceChart.data.labels = datesSlice;
        performanceChart.data.datasets[0].data = strategyDDSlice;
        performanceChart.data.datasets[1].data = benchmarkDDSlice;
        performanceChart.update('none');

    } else if (currentChartType === 'annual') {
        const currentYear = stockData.dates[index].substring(0, 4);
        const filteredStrategyAnnual = backtestResults.annualReturns.filter(a => a.year <= currentYear);
        const filteredBenchmarkAnnual = benchmarkResults.annualReturns.filter(a => a.year <= currentYear);

        performanceChart.data.labels = filteredStrategyAnnual.map(a => a.year);
        performanceChart.data.datasets[0].data = filteredStrategyAnnual.map(a => a.returnRate.toFixed(2));
        performanceChart.data.datasets[1].data = filteredBenchmarkAnnual.map(a => a.returnRate.toFixed(2));
        performanceChart.update('none');
    }

    // 2. Update allocation doughnut chart
    const dayHoldings = backtestResults.dailySnapshots[index].holdings;

    const colors = [
        '#10B981', '#3B82F6', '#EF4444', '#F59E0B', '#8B5CF6',
        '#EC4899', '#14B8A6', '#F97316', '#06B6D4', '#84CC16',
        '#10B981', '#3B82F6', '#EF4444', '#F59E0B', '#8B5CF6',
        '#EC4899', '#14B8A6', '#F97316', '#06B6D4', '#84CC16'
    ];

    const chartLabels = dayHoldings.map(a => a.ticker);
    const chartData = dayHoldings.map(a => a.percentage.toFixed(2));
    const backgroundColors = dayHoldings.map((a, idx) => colors[idx % colors.length]);

    DOM.allocationLegend.innerHTML = dayHoldings.map((a, idx) => `
        <div class="legend-item">
            <span class="legend-color" style="background-color: ${backgroundColors[idx]}"></span>
            <span class="legend-label">${a.ticker}</span>
            <span class="legend-val">${a.percentage.toFixed(1)}%</span>
        </div>
    `).join('');

    allocationChart.data.labels = chartLabels;
    allocationChart.data.datasets[0].data = chartData;
    allocationChart.data.datasets[0].backgroundColor = backgroundColors;
    allocationChart.update('none');

    // 3. Update trades log table up to this day
    const playbackDate = stockData.dates[index];
    const logsUpToDate = backtestResults.tradeLog.filter(t => t.date <= playbackDate);

    if (logsUpToDate.length === 0) {
        DOM.tradeLogBody.innerHTML = `
            <tr>
                <td colspan="6" class="empty-table">No trades recorded up to this date.</td>
            </tr>
        `;
    } else {
        const recentTradesSlice = logsUpToDate.slice(-30).reverse();
        DOM.tradeLogBody.innerHTML = recentTradesSlice.map(trade => {
            const actionBadgeClass = trade.action === 'BUY' ? 'badge badge-buy' : 'badge badge-sell';
            return `
                <tr>
                    <td>${trade.date}</td>
                    <td><strong>${trade.ticker}</strong></td>
                    <td><span class="${actionBadgeClass}">${trade.action}</span></td>
                    <td>$${trade.price.toFixed(2)}</td>
                    <td>${trade.shares.toFixed(2)}</td>
                    <td>$${trade.value.toLocaleString(undefined, { minimumFractionDigits: 2, maximumFractionDigits: 2 })}</td>
                </tr>
            `;
        }).join('');
    }
}

// --- Update UI KPI Elements ---
function updateKPIs(strat, bench) {
    DOM.metricTotalReturn.textContent = strat.totalReturn.toFixed(2) + '%';
    DOM.metricCAGR.textContent = strat.cagr.toFixed(2) + '%';
    DOM.metricDrawdown.textContent = strat.maxDrawdown.toFixed(2) + '%';
    DOM.metricSharpe.textContent = strat.sharpe.toFixed(2);

    if (strat.totalReturn >= 0) {
        DOM.metricTotalReturn.className = 'kpi-value positive';
    } else {
        DOM.metricTotalReturn.className = 'kpi-value negative';
    }

    DOM.benchmarkTotalReturn.textContent = 'DJIA Index: ' + bench.totalReturn.toFixed(2) + '%';
    DOM.benchmarkCAGR.textContent = 'DJIA Index: ' + bench.cagr.toFixed(2) + '%';
    DOM.benchmarkDrawdown.textContent = 'DJIA Index: ' + bench.maxDrawdown.toFixed(2) + '%';
    DOM.benchmarkSharpe.textContent = 'DJIA Index: ' + bench.sharpe.toFixed(2);
}

// --- Render Trade Logs ---
function renderTradeLog(results) {
    const log = results.tradeLog;

    if (log.length === 0) {
        DOM.tradeLogBody.innerHTML = `
            <tr>
                <td colspan="6" class="empty-table">No recent trades recorded for this rebalancing configuration.</td>
            </tr>
        `;
        return;
    }

    const recentTrades = log.slice(-30).reverse();

    DOM.tradeLogBody.innerHTML = recentTrades.map(trade => {
        const actionBadgeClass = trade.action === 'BUY' ? 'badge badge-buy' : 'badge badge-sell';
        return `
            <tr>
                <td>${trade.date}</td>
                <td><strong>${trade.ticker}</strong></td>
                <td><span class="${actionBadgeClass}">${trade.action}</span></td>
                <td>$${trade.price.toFixed(2)}</td>
                <td>${trade.shares.toFixed(2)}</td>
                <td>$${trade.value.toLocaleString(undefined, { minimumFractionDigits: 2, maximumFractionDigits: 2 })}</td>
            </tr>
        `;
    }).join('');
}

// --- Render Portfolio Allocation Pie & Legend ---
function renderAllocationChart(results) {
    const allocations = results.finalAllocations;

    if (allocationChart) {
        allocationChart.destroy();
    }

    if (allocations.length === 0) {
        DOM.allocationLegend.innerHTML = '<span class="legend-item" style="grid-column: 1/-1; text-align: center; color: var(--text-muted)">100% Cash / Liquidated</span>';
        return;
    }

    const colors = [
        '#10B981', '#3B82F6', '#EF4444', '#F59E0B', '#8B5CF6',
        '#EC4899', '#14B8A6', '#F97316', '#06B6D4', '#84CC16',
        '#10B981', '#3B82F6', '#EF4444', '#F59E0B', '#8B5CF6',
        '#EC4899', '#14B8A6', '#F97316', '#06B6D4', '#84CC16'
    ];

    const chartLabels = allocations.map(a => a.ticker);
    const chartData = allocations.map(a => a.percentage.toFixed(2));
    const backgroundColors = allocations.map((a, idx) => colors[idx % colors.length]);

    DOM.allocationLegend.innerHTML = allocations.map((a, idx) => `
        <div class="legend-item">
            <span class="legend-color" style="background-color: ${backgroundColors[idx]}"></span>
            <span class="legend-label">${a.ticker}</span>
            <span class="legend-val">${a.percentage.toFixed(1)}%</span>
        </div>
    `).join('');

    allocationChart = new Chart(DOM.allocationCanvas, {
        type: 'doughnut',
        data: {
            labels: chartLabels,
            datasets: [{
                data: chartData,
                backgroundColor: backgroundColors,
                borderWidth: 1,
                borderColor: '#111827'
            }]
        },
        options: {
            responsive: true,
            maintainAspectRatio: false,
            plugins: {
                legend: { display: false },
                tooltip: {
                    callbacks: {
                        label: function(context) {
                            return ` ${context.label}: ${context.raw}%`;
                        }
                    }
                }
            },
            cutout: '65%'
        }
    });
}

// --- Render Performance Charts ---
function renderMainChart() {
    if (!backtestResults || !benchmarkResults) return;

    if (performanceChart) {
        performanceChart.destroy();
    }

    const dates = stockData.dates;
    let chartConfig = {};

    if (currentChartType === 'performance') {
        const strategyCurve = backtestResults.equityCurve.map(p => p.value);
        const benchmarkCurve = benchmarkResults.equityCurve.map(p => p.value);

        chartConfig = {
            type: 'line',
            data: {
                labels: dates,
                datasets: [
                    {
                        label: 'Algorithmic Strategy',
                        data: strategyCurve,
                        borderColor: '#10B981',
                        borderWidth: 2,
                        fill: false,
                        pointRadius: 0,
                        pointHoverRadius: 5,
                        tension: 0.1
                    },
                    {
                        label: 'DJIA Benchmark Index',
                        data: benchmarkCurve,
                        borderColor: '#3B82F6',
                        borderWidth: 1.5,
                        fill: false,
                        pointRadius: 0,
                        pointHoverRadius: 5,
                        borderDash: [5, 5],
                        tension: 0.1
                    }
                ]
            },
            options: {
                responsive: true,
                maintainAspectRatio: false,
                interaction: {
                    mode: 'index',
                    intersect: false
                },
                plugins: {
                    legend: {
                        position: 'top',
                        labels: {
                            color: '#9CA3AF',
                            font: { family: 'Inter', weight: 500 }
                        }
                    },
                    tooltip: {
                        callbacks: {
                            label: function(context) {
                                return ` ${context.dataset.label}: $${Math.round(context.raw).toLocaleString()}`;
                            }
                        }
                    }
                },
                scales: {
                    x: {
                        grid: { color: 'rgba(255, 255, 255, 0.03)' },
                        ticks: { color: '#6B7280', maxTicksLimit: 12 }
                    },
                    y: {
                        grid: { color: 'rgba(255, 255, 255, 0.05)' },
                        ticks: {
                            color: '#9CA3AF',
                            callback: function(value) { return '$' + value.toLocaleString(); }
                        }
                    }
                }
            }
        };

    } else if (currentChartType === 'drawdown') {
        const strategyDD = backtestResults.drawdowns.map(d => d.drawdown);
        const benchmarkDD = benchmarkResults.drawdowns.map(d => d.drawdown);

        chartConfig = {
            type: 'line',
            data: {
                labels: dates,
                datasets: [
                    {
                        label: 'Strategy Drawdown',
                        data: strategyDD,
                        borderColor: '#EF4444',
                        backgroundColor: 'rgba(239, 68, 68, 0.1)',
                        borderWidth: 1.5,
                        fill: true,
                        pointRadius: 0,
                        pointHoverRadius: 5,
                        tension: 0.1
                    },
                    {
                        label: 'DJIA Benchmark Drawdown',
                        data: benchmarkDD,
                        borderColor: 'rgba(59, 130, 246, 0.5)',
                        borderWidth: 1,
                        fill: false,
                        pointRadius: 0,
                        pointHoverRadius: 5,
                        borderDash: [3, 3],
                        tension: 0.1
                    }
                ]
            },
            options: {
                responsive: true,
                maintainAspectRatio: false,
                interaction: {
                    mode: 'index',
                    intersect: false
                },
                plugins: {
                    legend: {
                        position: 'top',
                        labels: {
                            color: '#9CA3AF',
                            font: { family: 'Inter', weight: 500 }
                        }
                    },
                    tooltip: {
                        callbacks: {
                            label: function(context) { return ` ${context.dataset.label}: ${context.raw.toFixed(2)}%`; }
                        }
                    }
                },
                scales: {
                    x: {
                        grid: { color: 'rgba(255, 255, 255, 0.03)' },
                        ticks: { color: '#6B7280', maxTicksLimit: 12 }
                    },
                    y: {
                        grid: { color: 'rgba(255, 255, 255, 0.05)' },
                        ticks: {
                            color: '#9CA3AF',
                            callback: function(value) { return value + '%'; }
                        },
                        max: 0
                    }
                }
            }
        };

    } else if (currentChartType === 'annual') {
        const strategyAnnual = backtestResults.annualReturns;
        const benchmarkAnnual = benchmarkResults.annualReturns;

        const years = strategyAnnual.map(a => a.year);
        const strategyData = strategyAnnual.map(a => a.returnRate.toFixed(2));
        const benchmarkData = benchmarkAnnual.map(a => a.returnRate.toFixed(2));

        chartConfig = {
            type: 'bar',
            data: {
                labels: years,
                datasets: [
                    {
                        label: 'Strategy Return',
                        data: strategyData,
                        backgroundColor: '#10B981',
                        borderRadius: 4
                    },
                    {
                        label: 'DJIA Benchmark Return',
                        data: benchmarkData,
                        backgroundColor: '#3B82F6',
                        borderRadius: 4
                    }
                ]
            },
            options: {
                responsive: true,
                maintainAspectRatio: false,
                plugins: {
                    legend: {
                        position: 'top',
                        labels: {
                            color: '#9CA3AF',
                            font: { family: 'Inter', weight: 500 }
                        }
                    },
                    tooltip: {
                        callbacks: {
                            label: function(context) { return ` ${context.dataset.label}: ${context.raw}%`; }
                        }
                    }
                },
                scales: {
                    x: {
                        grid: { color: 'rgba(255, 255, 255, 0.03)' },
                        ticks: { color: '#9CA3AF' }
                    },
                    y: {
                        grid: { color: 'rgba(255, 255, 255, 0.05)' },
                        ticks: {
                            color: '#9CA3AF',
                            callback: function(value) { return value + '%'; }
                        }
                    }
                }
            }
        };
    }

    performanceChart = new Chart(DOM.chartCanvas, chartConfig);
}
"""

with open('index.html', 'w') as f: f.write(html_code)
with open('style.css', 'w') as f: f.write(css_code)
with open('app.js', 'w') as f: f.write(js_code)

# Start HTTP server on thread
class ThreadedHTTPServer(object):
    def __init__(self, port):
        self.port = port
        handler = http.server.SimpleHTTPRequestHandler
        self.server = socketserver.TCPServer(("", self.port), handler)
        self.thread = threading.Thread(target=self.server.serve_forever)
        self.thread.daemon = True
        self.thread.start()
    def shutdown(self):
        self.server.shutdown()

port = portpicker.pick_unused_port()
server = ThreadedHTTPServer(port)
print(f"Local dashboard server started on port {port}")

# Serve the local dashboard inside Google Colab
output.serve_kernel_port_as_iframe(port, height=750)

Local dashboard server started on port 34581


<IPython.core.display.Javascript object>

##6.CONCLUSIONS

This notebook has served as a comprehensive demonstration of ANTIGRAVITY's prowess in enabling the rapid development, rigorous backtesting, and intuitive visualization of algorithmic trading strategies. From the initial conceptualization of strategies like 'Dogs of the Dow' and 'Momentum' to their meticulous execution within a pure Python backtesting engine, and finally to the interactive, real-time playback experience of the embedded web dashboard, we have showcased a full spectrum of capabilities that empower quantitative traders and financial analysts.

We began by establishing the foundation: acquiring historical market data for the 30 components of the Dow Jones Industrial Average, ensuring accuracy with adjusted close prices and dividend distributions. This data forms the bedrock upon which all subsequent simulations are built, emphasizing the importance of clean and reliable inputs for meaningful analysis. The subsequent explanation of ANTIGRAVITY's generative approach highlighted how complex requirements can be translated into functional code, spanning Python for the simulation logic and HTML/CSS/JavaScript for the user interface, thus streamlining the development workflow considerably.

The core of this demonstration lies in the robust pure Python backtesting engine. We delved into its architecture, illustrating how it systematically simulates daily market activity, handles rebalancing events, executes trades with defined transaction costs, and meticulously tracks portfolio value. This engine's output—equity curves, drawdowns, trade logs, and daily snapshots—forms the quantitative evidence necessary for evaluating strategy performance against a benchmark. The `calculate_metrics` function, in particular, distills this rich data into key performance indicators such as CAGR, Sharpe Ratio, and Maximum Drawdown, providing a concise yet powerful summary of a strategy's historical efficacy and risk profile.

Perhaps the most engaging aspect of this notebook is the interactive web dashboard. By integrating a dynamic HTML/CSS/JS application directly into the Colab environment, ANTIGRAVITY transforms static backtest results into an immersive experience. Users can visually track the evolution of their portfolio, observe rebalancing events unfold through the 'Dynamic Portfolio Composition Log Feed,' and witness the real-time impact of market fluctuations and strategy decisions. This level of interactive visualization is not merely aesthetic; it fosters deeper intuition, facilitates the identification of subtle patterns, and allows for a more nuanced understanding of strategy behavior than static reports alone. The ability to adjust parameters on-the-fly and immediately re-run simulations underscores the platform's flexibility and power.

In conclusion, this notebook exemplifies how ANTIGRAVITY empowers users to transcend traditional analytical limitations. It provides a robust, transparent, and interactive environment for exploring the vast landscape of algorithmic trading strategies. Whether for research, education, or practical application, the synergy of a powerful backtesting engine and a dynamic visualization dashboard makes ANTIGRAVITY an invaluable tool for anyone looking to gain an edge in the complex world of financial markets. The future of algorithmic trading lies in platforms that offer both deep analytical power and intuitive user engagement, and this demonstration unequivocally proves ANTIGRAVITY's position at the forefront of this evolution.